# OpenPlaque v0.3 — Standalone Boundary Refinement Parameter Tuning

This notebook is designed for a **fresh Colab/runtime**. It installs requirements, clones OpenPlaque, writes the v0.3 tuning modules into the environment, configures nnU-Net, loads the CCTA DICOM ZIP and model ZIP from Google Drive, runs segmentation, tunes boundary-refinement parameters, and writes HTML/CSV/JSON reports.

Expected Drive files by default:

```text
/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip
/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip
```

Research use only. Not for clinical decision-making.


In [ ]:
# 0. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1. User-editable paths
from pathlib import Path

STUDY_ZIP = Path('/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip')
MODEL_ZIP = Path('/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip')
SAVE_DIR = Path('/content/drive/MyDrive/OpenPlaque/Segmentations')

REPO_DIR = Path('/content/OpenPlaque')
SRC_DIR = REPO_DIR / 'src'

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Study ZIP:', STUDY_ZIP, STUDY_ZIP.exists())
print('Model ZIP:', MODEL_ZIP, MODEL_ZIP.exists())
print('Save dir:', SAVE_DIR)


In [ ]:
# 2. Fresh-environment install and repository setup
# This cell is intentionally self-contained for a new Colab runtime.
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt

# Extra packages used by the tuning/report workflow. Most may already be present.
!pip install -q pandas matplotlib scipy scikit-image SimpleITK pydicom nnunetv2


In [ ]:
# 3. Install bundled OpenPlaque v0.3 modules into the fresh runtime
import base64, sys, os
from pathlib import Path

REPO_DIR = Path('/content/OpenPlaque')
SRC_DIR = REPO_DIR / 'src'
PKG_DIR = SRC_DIR / 'openplaque'
PKG_DIR.mkdir(parents=True, exist_ok=True)

MODULES_B64 = {
    'artery_detection.py': 'IiIiCkF1dG9tYXRpYyBjb3JvbmFyeSBhcnRlcnkgY3VydmVkLXJlZm9ybWF0IHNlcmllcyBkZXRlY3Rpb24gZm9yIE9wZW5QbGFxdWUuCgpUaGlzIG1vZHVsZSByZXBsYWNlcyBoYXJkLWNvZGVkIHNlcmllcyBudW1iZXJzIHdpdGggbWV0YWRhdGEtYmFzZWQgaGV1cmlzdGljcy4KSXQgYWNjZXB0cyBhbiBPcGVuUGxhcXVlU3R1ZHktbGlrZSBvYmplY3QgYW5kIGNhbiBvcHRpb25hbGx5IGluc3BlY3QgYSBESUNPTSBaSVAKd2hlbiBtZXRhZGF0YSBpcyBub3QgZXhwb3NlZCBieSB0aGUgc3R1ZHkgb2JqZWN0LgoKUmVzZWFyY2ggdXNlIG9ubHkuIE5vdCBmb3IgY2xpbmljYWwgZGVjaXNpb24tbWFraW5nLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgRGljdCwgTGlzdCwgTWFwcGluZywgT3B0aW9uYWwsIFNlcXVlbmNlLCBUdXBsZQppbXBvcnQgcmUKaW1wb3J0IHRlbXBmaWxlCmltcG9ydCB6aXBmaWxlCgoKQVJURVJZX0FMSUFTRVM6IE1hcHBpbmdbc3RyLCBUdXBsZVtzdHIsIC4uLl1dID0gewogICAgIkxBRCI6ICgibGFkIiwgImxlZnQgYW50ZXJpb3IgZGVzY2VuZGluZyIpLAogICAgIlJDQSI6ICgicmNhIiwgInJpZ2h0IGNvcm9uYXJ5IiksCiAgICAiTENYIjogKCJsY3giLCAiY3giLCAiY2lyY3VtZmxleCIsICJsZWZ0IGNpcmN1bWZsZXgiKSwKfQoKQ1VSVkVEX0tFWVdPUkRTID0gKAogICAgImN1cnZlZCIsICJjcHIiLCAiY3VydmVkIHBsYW5hciIsICJyZWZvcm1hdCIsICJyZWZvcm1hdGlvbiIsICJtcHIiLAogICAgInZlc3NlbCIsICJjb3JvbmFyeSIsICJzdHJhaWdodGVuZWQiLCAic3RyZXRjaGVkIiwKKQoKCkBkYXRhY2xhc3MKY2xhc3MgQXJ0ZXJ5U2VyaWVzQ2FuZGlkYXRlOgogICAgYXJ0ZXJ5OiBzdHIKICAgIHNlcmllc19udW1iZXI6IGludAogICAgc2NvcmU6IGZsb2F0CiAgICBkZXNjcmlwdGlvbjogc3RyID0gIiIKICAgIHJlYXNvbjogc3RyID0gIiIKCgpkZWYgX25vcm1hbGlzZV90ZXh0KHZhbHVlOiBBbnkpIC0+IHN0cjoKICAgIHJldHVybiByZS5zdWIociJccysiLCAiICIsIHN0cih2YWx1ZSBvciAiIikpLnN0cmlwKCkubG93ZXIoKQoKCmRlZiBfZ2V0X2F0dHIob2JqOiBBbnksIG5hbWVzOiBTZXF1ZW5jZVtzdHJdLCBkZWZhdWx0OiBBbnkgPSBOb25lKSAtPiBBbnk6CiAgICBmb3IgbmFtZSBpbiBuYW1lczoKICAgICAgICBpZiBpc2luc3RhbmNlKG9iaiwgTWFwcGluZykgYW5kIG5hbWUgaW4gb2JqOgogICAgICAgICAgICByZXR1cm4gb2JqW25hbWVdCiAgICAgICAgaWYgaGFzYXR0cihvYmosIG5hbWUpOgogICAgICAgICAgICByZXR1cm4gZ2V0YXR0cihvYmosIG5hbWUpCiAgICByZXR1cm4gZGVmYXVsdAoKCmRlZiBfc2VyaWVzX251bWJlcl9mcm9tX3JlY29yZChyZWNvcmQ6IEFueSkgLT4gT3B0aW9uYWxbaW50XToKICAgIHZhbHVlID0gX2dldF9hdHRyKHJlY29yZCwgKCJzZXJpZXNfbnVtYmVyIiwgIlNlcmllc051bWJlciIsICJudW1iZXIiLCAiaWQiLCAic2VyaWVzIikpCiAgICBpZiB2YWx1ZSBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGludCh2YWx1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgZGlnaXRzID0gcmUuZmluZGFsbChyIlxkKyIsIHN0cih2YWx1ZSkpCiAgICAgICAgcmV0dXJuIGludChkaWdpdHNbLTFdKSBpZiBkaWdpdHMgZWxzZSBOb25lCgoKZGVmIF9kZXNjcmlwdGlvbl9mcm9tX3JlY29yZChyZWNvcmQ6IEFueSkgLT4gc3RyOgogICAgZmllbGRzID0gWwogICAgICAgICJkZXNjcmlwdGlvbiIsICJTZXJpZXNEZXNjcmlwdGlvbiIsICJzZXJpZXNfZGVzY3JpcHRpb24iLCAibmFtZSIsCiAgICAgICAgIlByb3RvY29sTmFtZSIsICJwcm90b2NvbF9uYW1lIiwgIkltYWdlVHlwZSIsICJpbWFnZV90eXBlIiwKICAgICAgICAiQm9keVBhcnRFeGFtaW5lZCIsICJib2R5X3BhcnRfZXhhbWluZWQiLAogICAgXQogICAgcGFydHMgPSBbXQogICAgZm9yIGZpZWxkIGluIGZpZWxkczoKICAgICAgICB2YWx1ZSA9IF9nZXRfYXR0cihyZWNvcmQsIChmaWVsZCwpKQogICAgICAgIGlmIHZhbHVlOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAobGlzdCwgdHVwbGUpKToKICAgICAgICAgICAgICAgIHBhcnRzLmFwcGVuZCgiICIuam9pbihtYXAoc3RyLCB2YWx1ZSkpKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcGFydHMuYXBwZW5kKHN0cih2YWx1ZSkpCiAgICByZXR1cm4gIiB8ICIuam9pbihwYXJ0cykKCgpkZWYgX3JlY29yZHNfZnJvbV9zdHVkeV9hdHRyaWJ1dGVzKHN0dWR5OiBBbnkpIC0+IExpc3RbQW55XToKICAgIGZvciBhdHRyIGluICgic2VyaWVzIiwgInNlcmllc19pbmZvIiwgInNlcmllc19tZXRhZGF0YSIsICJkaWNvbV9zZXJpZXMiLCAic2VyaWVzX3JlY29yZHMiKToKICAgICAgICB2YWx1ZSA9IGdldGF0dHIoc3R1ZHksIGF0dHIsIE5vbmUpCiAgICAgICAgaWYgdmFsdWU6CiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodmFsdWUsIE1hcHBpbmcpOgogICAgICAgICAgICAgICAgcmVjb3JkcyA9IFtdCiAgICAgICAgICAgICAgICBmb3Iga2V5LCByZWNvcmQgaW4gdmFsdWUuaXRlbXMoKToKICAgICAgICAgICAgICAgICAgICBpZiBpc2luc3RhbmNlKHJlY29yZCwgTWFwcGluZyk6CiAgICAgICAgICAgICAgICAgICAgICAgIG1lcmdlZCA9IGRpY3QocmVjb3JkKQogICAgICAgICAgICAgICAgICAgICAgICBtZXJnZWQuc2V0ZGVmYXVsdCgic2VyaWVzX251bWJlciIsIGtleSkKICAgICAgICAgICAgICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQobWVyZ2VkKQogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlY29yZHMuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgICAgIHJldHVybiByZWNvcmRzCiAgICAgICAgICAgIHJldHVybiBsaXN0KHZhbHVlKQoKICAgIGZvciBtZXRob2QgaW4gKCJsaXN0X3NlcmllcyIsICJnZXRfc2VyaWVzIiwgImF2YWlsYWJsZV9zZXJpZXMiLCAic2VyaWVzX3N1bW1hcnkiKToKICAgICAgICBmbiA9IGdldGF0dHIoc3R1ZHksIG1ldGhvZCwgTm9uZSkKICAgICAgICBpZiBjYWxsYWJsZShmbik6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHZhbHVlID0gZm4oKQogICAgICAgICAgICAgICAgaWYgdmFsdWU6CiAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgTWFwcGluZyk6CiAgICAgICAgICAgICAgICAgICAgICAgIG91dCA9IFtdCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBrZXksIHJlY29yZCBpbiB2YWx1ZS5pdGVtcygpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShyZWNvcmQsIE1hcHBpbmcpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1lcmdlZCA9IGRpY3QocmVjb3JkKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1lcmdlZC5zZXRkZWZhdWx0KCJzZXJpZXNfbnVtYmVyIiwga2V5KQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG91dC5hcHBlbmQobWVyZ2VkKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgICAgICAgICAgICAgcmV0dXJuIG91dAogICAgICAgICAgICAgICAgICAgIHJldHVybiBsaXN0KHZhbHVlKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgcGFzcwogICAgcmV0dXJuIFtdCgoKZGVmIF9kaWNvbV96aXBfcGF0aF9mcm9tX3N0dWR5KHN0dWR5OiBBbnkpIC0+IE9wdGlvbmFsW1BhdGhdOgogICAgZm9yIGF0dHIgaW4gKCJ6aXBfcGF0aCIsICJzdHVkeV96aXAiLCAiZGljb21femlwIiwgInBhdGgiLCAic291cmNlIiwgImlucHV0X3BhdGgiKToKICAgICAgICB2YWx1ZSA9IGdldGF0dHIoc3R1ZHksIGF0dHIsIE5vbmUpCiAgICAgICAgaWYgdmFsdWUgYW5kIHN0cih2YWx1ZSkubG93ZXIoKS5lbmRzd2l0aCgiLnppcCIpOgogICAgICAgICAgICBwID0gUGF0aChzdHIodmFsdWUpKQogICAgICAgICAgICBpZiBwLmV4aXN0cygpOgogICAgICAgICAgICAgICAgcmV0dXJuIHAKICAgIHJldHVybiBOb25lCgoKZGVmIF9yZWNvcmRzX2Zyb21fZGljb21femlwKHppcF9wYXRoOiBQYXRoLCBtYXhfZmlsZXM6IGludCA9IDI1MCkgLT4gTGlzdFtkaWN0XToKICAgICIiIlJlYWQgYSBzbWFsbCBzYW1wbGUgb2YgRElDT00gbWV0YWRhdGEgZnJvbSBhIFpJUCwgZ3JvdXBlZCBieSBTZXJpZXNOdW1iZXIuIiIiCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHB5ZGljb20KICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgcmV0dXJuIFtdCgogICAgcmVjb3Jkc19ieV9zZXJpZXM6IERpY3RbaW50LCBkaWN0XSA9IHt9CiAgICB3aXRoIHRlbXBmaWxlLlRlbXBvcmFyeURpcmVjdG9yeSgpIGFzIHRtcDoKICAgICAgICB0bXBkaXIgPSBQYXRoKHRtcCkKICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZSh6aXBfcGF0aCkgYXMgemY6CiAgICAgICAgICAgIG5hbWVzID0gW24gZm9yIG4gaW4gemYubmFtZWxpc3QoKSBpZiBub3Qgbi5lbmRzd2l0aCgiLyIpXQogICAgICAgICAgICAjIFByZWZlciBsaWtlbHkgRElDT00gZmlsZXMgYnV0IGtlZXAgdGhpcyBwZXJtaXNzaXZlLgogICAgICAgICAgICBuYW1lcyA9IG5hbWVzWzptYXhfZmlsZXNdCiAgICAgICAgICAgIGZvciBuYW1lIGluIG5hbWVzOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHRhcmdldCA9IHRtcGRpciAvIFBhdGgobmFtZSkubmFtZQogICAgICAgICAgICAgICAgICAgIHRhcmdldC53cml0ZV9ieXRlcyh6Zi5yZWFkKG5hbWUpKQogICAgICAgICAgICAgICAgICAgIGRzID0gcHlkaWNvbS5kY21yZWFkKHN0cih0YXJnZXQpLCBzdG9wX2JlZm9yZV9waXhlbHM9VHJ1ZSwgZm9yY2U9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICBzbiA9IGdldGF0dHIoZHMsICJTZXJpZXNOdW1iZXIiLCBOb25lKQogICAgICAgICAgICAgICAgICAgIGlmIHNuIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgc24gPSBpbnQoc24pCiAgICAgICAgICAgICAgICAgICAgaWYgc24gbm90IGluIHJlY29yZHNfYnlfc2VyaWVzOgogICAgICAgICAgICAgICAgICAgICAgICByZWNvcmRzX2J5X3Nlcmllc1tzbl0gPSB7CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAic2VyaWVzX251bWJlciI6IHNuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIlNlcmllc0Rlc2NyaXB0aW9uIjogZ2V0YXR0cihkcywgIlNlcmllc0Rlc2NyaXB0aW9uIiwgIiIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIlByb3RvY29sTmFtZSI6IGdldGF0dHIoZHMsICJQcm90b2NvbE5hbWUiLCAiIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiSW1hZ2VUeXBlIjogIiAiLmpvaW4obWFwKHN0ciwgZ2V0YXR0cihkcywgIkltYWdlVHlwZSIsIFtdKSkpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIkJvZHlQYXJ0RXhhbWluZWQiOiBnZXRhdHRyKGRzLCAiQm9keVBhcnRFeGFtaW5lZCIsICIiKSwKICAgICAgICAgICAgICAgICAgICAgICAgfQogICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgcmV0dXJuIGxpc3QocmVjb3Jkc19ieV9zZXJpZXMudmFsdWVzKCkpCgoKZGVmIF9yZWNvcmRzX2Zyb21fc3R1ZHkoc3R1ZHk6IEFueSkgLT4gTGlzdFtBbnldOgogICAgcmVjb3JkcyA9IF9yZWNvcmRzX2Zyb21fc3R1ZHlfYXR0cmlidXRlcyhzdHVkeSkKICAgIGlmIHJlY29yZHM6CiAgICAgICAgcmV0dXJuIHJlY29yZHMKICAgIHppcF9wYXRoID0gX2RpY29tX3ppcF9wYXRoX2Zyb21fc3R1ZHkoc3R1ZHkpCiAgICBpZiB6aXBfcGF0aDoKICAgICAgICByZXR1cm4gX3JlY29yZHNfZnJvbV9kaWNvbV96aXAoemlwX3BhdGgpCiAgICByZXR1cm4gW10KCgpkZWYgc2NvcmVfc2VyaWVzX2Zvcl9hcnRlcnkocmVjb3JkOiBBbnksIGFydGVyeTogc3RyKSAtPiBBcnRlcnlTZXJpZXNDYW5kaWRhdGU6CiAgICBhcnRlcnkgPSBhcnRlcnkudXBwZXIoKQogICAgbnVtYmVyID0gX3Nlcmllc19udW1iZXJfZnJvbV9yZWNvcmQocmVjb3JkKQogICAgZGVzY3JpcHRpb24gPSBfZGVzY3JpcHRpb25fZnJvbV9yZWNvcmQocmVjb3JkKQogICAgdGV4dCA9IF9ub3JtYWxpc2VfdGV4dChkZXNjcmlwdGlvbikKCiAgICBzY29yZSA9IDAuMAogICAgcmVhc29uczogTGlzdFtzdHJdID0gW10KCiAgICBmb3IgYWxpYXMgaW4gQVJURVJZX0FMSUFTRVNbYXJ0ZXJ5XToKICAgICAgICBhbGlhc190ZXh0ID0gX25vcm1hbGlzZV90ZXh0KGFsaWFzKQogICAgICAgIGlmIHJlLnNlYXJjaChyZiIoXnxbXmEtejAtOV0pe3JlLmVzY2FwZShhbGlhc190ZXh0KX0oW15hLXowLTldfCQpIiwgdGV4dCk6CiAgICAgICAgICAgIHNjb3JlICs9IDEwMAogICAgICAgICAgICByZWFzb25zLmFwcGVuZChmIm1hdGNoZWQgYXJ0ZXJ5IGFsaWFzICd7YWxpYXN9JyIpCiAgICAgICAgICAgIGJyZWFrCgogICAgY3VydmVkX2hpdHMgPSBba3cgZm9yIGt3IGluIENVUlZFRF9LRVlXT1JEUyBpZiBrdyBpbiB0ZXh0XQogICAgaWYgY3VydmVkX2hpdHM6CiAgICAgICAgc2NvcmUgKz0gbWluKDMwLCAxMCAqIGxlbihjdXJ2ZWRfaGl0cykpCiAgICAgICAgcmVhc29ucy5hcHBlbmQoImN1cnZlZC9jb3JvbmFyeSBrZXl3b3JkIG1hdGNoIikKCiAgICBpZiBudW1iZXIgYW5kIG51bWJlciA+PSAxMDAwOgogICAgICAgIHNjb3JlICs9IDUKICAgICAgICByZWFzb25zLmFwcGVuZCgiaGlnaCBkZXJpdmVkL3JlZm9ybWF0IHNlcmllcyBudW1iZXIiKQoKICAgIHJldHVybiBBcnRlcnlTZXJpZXNDYW5kaWRhdGUoCiAgICAgICAgYXJ0ZXJ5PWFydGVyeSwKICAgICAgICBzZXJpZXNfbnVtYmVyPW51bWJlciBpZiBudW1iZXIgaXMgbm90IE5vbmUgZWxzZSAtMSwKICAgICAgICBzY29yZT1zY29yZSwKICAgICAgICBkZXNjcmlwdGlvbj1kZXNjcmlwdGlvbiwKICAgICAgICByZWFzb249IjsgIi5qb2luKHJlYXNvbnMpIG9yICJubyBzdHJvbmcgbWV0YWRhdGEgbWF0Y2giLAogICAgKQoKCmRlZiBkZXRlY3RfYXJ0ZXJ5X3NlcmllcygKICAgIHN0dWR5OiBBbnksCiAgICByZXF1aXJlZDogU2VxdWVuY2Vbc3RyXSA9ICgiTEFEIiwgIlJDQSIsICJMQ1giKSwKICAgIGZhbGxiYWNrX3NlcmllczogT3B0aW9uYWxbTWFwcGluZ1tzdHIsIGludF1dID0gTm9uZSwKICAgIG1pbl9zY29yZTogZmxvYXQgPSAxMDAuMCwKICAgIHJldHVybl9jYW5kaWRhdGVzOiBib29sID0gRmFsc2UsCik6CiAgICAiIiIKICAgIERldGVjdCBsaWtlbHkgTEFEL1JDQS9MQ1ggY3VydmVkLXJlZm9ybWF0IHNlcmllcyBudW1iZXJzLgoKICAgIElmIG1ldGFkYXRhIGlzIHVuYXZhaWxhYmxlIG9yIGFtYmlndW91cywgYGZhbGxiYWNrX3Nlcmllc2AgaXMgdXNlZC4gRm9yIHRoZQogICAgVUNMQSBleGFtcGxlLCBwYXNzIHsiUkNBIjogMTAzNSwgIkxDWCI6IDEwMzksICJMQUQiOiAxMDQzfS4KICAgICIiIgogICAgcmVjb3JkcyA9IF9yZWNvcmRzX2Zyb21fc3R1ZHkoc3R1ZHkpCiAgICBmYWxsYmFjayA9IHtrLnVwcGVyKCk6IGludCh2KSBmb3IgaywgdiBpbiAoZmFsbGJhY2tfc2VyaWVzIG9yIHt9KS5pdGVtcygpfQogICAgZGV0ZWN0ZWQ6IERpY3Rbc3RyLCBpbnRdID0ge30KICAgIGFsbF9jYW5kaWRhdGVzOiBEaWN0W3N0ciwgTGlzdFtBcnRlcnlTZXJpZXNDYW5kaWRhdGVdXSA9IHt9CgogICAgZm9yIGFydGVyeSBpbiBbYS51cHBlcigpIGZvciBhIGluIHJlcXVpcmVkXToKICAgICAgICBjYW5kaWRhdGVzID0gW3Njb3JlX3Nlcmllc19mb3JfYXJ0ZXJ5KHIsIGFydGVyeSkgZm9yIHIgaW4gcmVjb3Jkc10KICAgICAgICBjYW5kaWRhdGVzID0gW2MgZm9yIGMgaW4gY2FuZGlkYXRlcyBpZiBjLnNlcmllc19udW1iZXIgPj0gMF0KICAgICAgICBjYW5kaWRhdGVzLnNvcnQoa2V5PWxhbWJkYSBjOiBjLnNjb3JlLCByZXZlcnNlPVRydWUpCiAgICAgICAgYWxsX2NhbmRpZGF0ZXNbYXJ0ZXJ5XSA9IGNhbmRpZGF0ZXNbOjEwXQoKICAgICAgICBpZiBjYW5kaWRhdGVzIGFuZCBjYW5kaWRhdGVzWzBdLnNjb3JlID49IG1pbl9zY29yZToKICAgICAgICAgICAgZGV0ZWN0ZWRbYXJ0ZXJ5XSA9IGNhbmRpZGF0ZXNbMF0uc2VyaWVzX251bWJlcgogICAgICAgIGVsaWYgYXJ0ZXJ5IGluIGZhbGxiYWNrOgogICAgICAgICAgICBkZXRlY3RlZFthcnRlcnldID0gZmFsbGJhY2tbYXJ0ZXJ5XQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHRvcCA9IGNhbmRpZGF0ZXNbMF0gaWYgY2FuZGlkYXRlcyBlbHNlIE5vbmUKICAgICAgICAgICAgZGV0YWlsID0gZiIgVG9wIGNhbmRpZGF0ZSB3YXMge3RvcC5zZXJpZXNfbnVtYmVyfSBzY29yZT17dG9wLnNjb3JlOi4xZn06IHt0b3AuZGVzY3JpcHRpb259IiBpZiB0b3AgZWxzZSAiIgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJDb3VsZCBub3QgY29uZmlkZW50bHkgZGV0ZWN0IHthcnRlcnl9IGN1cnZlZC1yZWZvcm1hdCBzZXJpZXMuIgogICAgICAgICAgICAgICAgZiIgUHJvdmlkZSBmYWxsYmFja19zZXJpZXMuIHtkZXRhaWx9IgogICAgICAgICAgICApCgogICAgaWYgcmV0dXJuX2NhbmRpZGF0ZXM6CiAgICAgICAgcmV0dXJuIGRldGVjdGVkLCBhbGxfY2FuZGlkYXRlcwogICAgcmV0dXJuIGRldGVjdGVkCgoKZGVmIGxvYWRfZGV0ZWN0ZWRfYXJ0ZXJpZXMoCiAgICBzdHVkeTogQW55LAogICAgZmFsbGJhY2tfc2VyaWVzOiBPcHRpb25hbFtNYXBwaW5nW3N0ciwgaW50XV0gPSBOb25lLAogICAgcmVxdWlyZWQ6IFNlcXVlbmNlW3N0cl0gPSAoIkxBRCIsICJSQ0EiLCAiTENYIiksCikgLT4gRGljdFtzdHIsIFR1cGxlW0FueSwgQW55LCBBbnldXToKICAgIHNlcmllc19tYXAgPSBkZXRlY3RfYXJ0ZXJ5X3NlcmllcyhzdHVkeSwgcmVxdWlyZWQ9cmVxdWlyZWQsIGZhbGxiYWNrX3Nlcmllcz1mYWxsYmFja19zZXJpZXMpCiAgICByZXR1cm4ge2FydGVyeTogc3R1ZHkubG9hZF9zZXJpZXMoc2VyaWVzX251bWJlcikgZm9yIGFydGVyeSwgc2VyaWVzX251bWJlciBpbiBzZXJpZXNfbWFwLml0ZW1zKCl9Cg==',
    'boundary.py': 'IiIiCk9wZW5QbGFxdWUgYm91bmRhcnkgcmVmaW5lbWVudC4KCkV4cGVyaW1lbnRhbCBwb3N0LXByb2Nlc3NpbmcgZm9yIG5uVS1OZXQgcGxhcXVlIG1hc2tzLgoKR29hbDoKICAgIFJlZHVjZSBmYWxzZS1wb3NpdGl2ZSBwbGFxdWUgYm91bmRhcnkgdm94ZWxzIHRoYXQgbWF5IHJlcHJlc2VudCBsdW1lbiwKICAgIG5vcm1hbCB2ZXNzZWwgd2FsbCwgaW50ZXJwb2xhdGlvbiBhcnRpZmFjdHMsIG9yIGlzb2xhdGVkIG5vaXNlLgoKSW1wb3J0YW50OgogICAgVGhpcyBpcyByZXNlYXJjaCBjb2RlIG9ubHkuIEl0IGhhcyBub3QgYmVlbiBjbGluaWNhbGx5IHZhbGlkYXRlZC4KIiIiCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkgaW1wb3J0IG5kaW1hZ2UgYXMgbmRpCgoKQGRhdGFjbGFzcwpjbGFzcyBSZWZpbmVtZW50UmVzdWx0OgogICAgb3JpZ2luYWxfbWFzazogbnAubmRhcnJheQogICAgcmVmaW5lZF9tYXNrOiBucC5uZGFycmF5CiAgICByZW1vdmVkX21hc2s6IG5wLm5kYXJyYXkKICAgIHNwYWNpbmc6IHR1cGxlCiAgICBtZXRob2Q6IHN0cgogICAgcGFyYW1ldGVyczogZGljdAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHZveGVsX3ZvbHVtZV9tbTMoc2VsZik6CiAgICAgICAgcmV0dXJuIGZsb2F0KG5wLnByb2Qoc2VsZi5zcGFjaW5nKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBvcmlnaW5hbF9wbGFxdWVfdm94ZWxzKHNlbGYpOgogICAgICAgIHJldHVybiBpbnQobnAuc3VtKHNlbGYub3JpZ2luYWxfbWFzayA9PSAyKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiByZWZpbmVkX3BsYXF1ZV92b3hlbHMoc2VsZik6CiAgICAgICAgcmV0dXJuIGludChucC5zdW0oc2VsZi5yZWZpbmVkX21hc2sgPT0gMikpCgogICAgQHByb3BlcnR5CiAgICBkZWYgcmVtb3ZlZF92b3hlbHMoc2VsZik6CiAgICAgICAgcmV0dXJuIGludChucC5zdW0oc2VsZi5yZW1vdmVkX21hc2spKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG9yaWdpbmFsX3Rwdl9tbTMoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYub3JpZ2luYWxfcGxhcXVlX3ZveGVscyAqIHNlbGYudm94ZWxfdm9sdW1lX21tMwoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHJlZmluZWRfdHB2X21tMyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5yZWZpbmVkX3BsYXF1ZV92b3hlbHMgKiBzZWxmLnZveGVsX3ZvbHVtZV9tbTMKCiAgICBAcHJvcGVydHkKICAgIGRlZiByZW1vdmVkX3ZvbHVtZV9tbTMoc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYucmVtb3ZlZF92b3hlbHMgKiBzZWxmLnZveGVsX3ZvbHVtZV9tbTMKCiAgICBkZWYgc3VtbWFyeShzZWxmKToKICAgICAgICBwcmludCgiQm91bmRhcnkgcmVmaW5lbWVudCIpCiAgICAgICAgcHJpbnQoIk1ldGhvZDoiLCBzZWxmLm1ldGhvZCkKICAgICAgICBwcmludCgiUGFyYW1ldGVyczoiLCBzZWxmLnBhcmFtZXRlcnMpCiAgICAgICAgcHJpbnQoKQogICAgICAgIHByaW50KGYiT3JpZ2luYWwgcGxhcXVlIHZveGVsczoge3NlbGYub3JpZ2luYWxfcGxhcXVlX3ZveGVsc30iKQogICAgICAgIHByaW50KGYiUmVmaW5lZCBwbGFxdWUgdm94ZWxzOiAge3NlbGYucmVmaW5lZF9wbGFxdWVfdm94ZWxzfSIpCiAgICAgICAgcHJpbnQoZiJSZW1vdmVkIHZveGVsczogICAgICAgICB7c2VsZi5yZW1vdmVkX3ZveGVsc30iKQogICAgICAgIHByaW50KCkKICAgICAgICBwcmludChmIk9yaWdpbmFsIFRQVjoge3NlbGYub3JpZ2luYWxfdHB2X21tMzouMmZ9IG1twrMiKQogICAgICAgIHByaW50KGYiUmVmaW5lZCBUUFY6ICB7c2VsZi5yZWZpbmVkX3Rwdl9tbTM6LjJmfSBtbcKzIikKICAgICAgICBwcmludChmIlJlbW92ZWQgdm9sOiAge3NlbGYucmVtb3ZlZF92b2x1bWVfbW0zOi4yZn0gbW3CsyIpCgogICAgZGVmIHNob3dfcmVtb3ZlZF9vdmVybGF5KHNlbGYsIHZvbHVtZSwgej1Ob25lLCB2bWluPS0yMDAsIHZtYXg9ODAwKToKICAgICAgICBpbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CgogICAgICAgIGlmIHogaXMgTm9uZToKICAgICAgICAgICAgY291bnRzID0gbnAuc3VtKHNlbGYucmVtb3ZlZF9tYXNrLCBheGlzPSgxLCAyKSkKICAgICAgICAgICAgeiA9IGludChucC5hcmdtYXgoY291bnRzKSkKCiAgICAgICAgcGx0LmZpZ3VyZShmaWdzaXplPSg4LCA4KSkKICAgICAgICBwbHQuaW1zaG93KHZvbHVtZVt6XSwgY21hcD0iZ3JheSIsIHZtaW49dm1pbiwgdm1heD12bWF4KQogICAgICAgIHBsdC5pbXNob3coc2VsZi5yZW1vdmVkX21hc2tbel0sIGFscGhhPTAuNikKICAgICAgICBwbHQudGl0bGUoZiJSZW1vdmVkIHBsYXF1ZS1ib3VuZGFyeSB2b3hlbHMsIHNsaWNlIHt6fSIpCiAgICAgICAgcGx0LmF4aXMoIm9mZiIpCiAgICAgICAgcGx0LnNob3coKQoKICAgIGRlZiBzaG93X3JlZmluZWRfb3ZlcmxheShzZWxmLCB2b2x1bWUsIHo9Tm9uZSwgdm1pbj0tMjAwLCB2bWF4PTgwMCk6CiAgICAgICAgaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAoKICAgICAgICBpZiB6IGlzIE5vbmU6CiAgICAgICAgICAgIGNvdW50cyA9IG5wLnN1bShzZWxmLnJlZmluZWRfbWFzayA9PSAyLCBheGlzPSgxLCAyKSkKICAgICAgICAgICAgeiA9IGludChucC5hcmdtYXgoY291bnRzKSkKCiAgICAgICAgcGx0LmZpZ3VyZShmaWdzaXplPSg4LCA4KSkKICAgICAgICBwbHQuaW1zaG93KHZvbHVtZVt6XSwgY21hcD0iZ3JheSIsIHZtaW49dm1pbiwgdm1heD12bWF4KQogICAgICAgIHBsdC5pbXNob3coc2VsZi5yZWZpbmVkX21hc2tbel0gPT0gMiwgYWxwaGE9MC42KQogICAgICAgIHBsdC50aXRsZShmIlJlZmluZWQgcGxhcXVlIG1hc2ssIHNsaWNlIHt6fSIpCiAgICAgICAgcGx0LmF4aXMoIm9mZiIpCiAgICAgICAgcGx0LnNob3coKQoKCmRlZiByZW1vdmVfc21hbGxfY29tcG9uZW50cyhtYXNrLCBtaW5fdm94ZWxzPTEwLCBwbGFxdWVfbGFiZWw9Mik6CiAgICAiIiIKICAgIFJlbW92ZSBzbWFsbCBkaXNjb25uZWN0ZWQgcGxhcXVlIGNvbXBvbmVudHMuCgogICAgVGhpcyBpcyB1c2VmdWwgZm9yIGVsaW1pbmF0aW5nIHRpbnkgc3BlY2tsZXMsIGJ1dCBpdCBkb2VzIG5vdCBhZGRyZXNzCiAgICBsdW1lbi93YWxsIGJvdW5kYXJ5IGVycm9ycy4KICAgICIiIgogICAgcGxhcXVlID0gbWFzayA9PSBwbGFxdWVfbGFiZWwKICAgIGxhYmVscywgbiA9IG5kaS5sYWJlbChwbGFxdWUpCgogICAgc2l6ZXMgPSBucC5iaW5jb3VudChsYWJlbHMucmF2ZWwoKSkKICAgIGtlZXBfbGFiZWxzID0gbnAud2hlcmUoc2l6ZXMgPj0gbWluX3ZveGVscylbMF0KICAgIGtlZXBfbGFiZWxzID0ga2VlcF9sYWJlbHNba2VlcF9sYWJlbHMgIT0gMF0KCiAgICBrZWVwID0gbnAuaXNpbihsYWJlbHMsIGtlZXBfbGFiZWxzKQoKICAgIHJlZmluZWQgPSBtYXNrLmNvcHkoKQogICAgcmVmaW5lZFtwbGFxdWUgJiB+a2VlcF0gPSAwCgogICAgcmV0dXJuIHJlZmluZWQKCgpkZWYgZXJvZGVfcGxhcXVlX2JvdW5kYXJ5KG1hc2ssIGl0ZXJhdGlvbnM9MSwgcGxhcXVlX2xhYmVsPTIpOgogICAgIiIiCiAgICBDb25zZXJ2YXRpdmUgY29yZS1wbGFxdWUgbWFzayB1c2luZyBiaW5hcnkgZXJvc2lvbi4KCiAgICBUaGlzIGVzdGltYXRlcyBhIGhpZ2gtY29uZmlkZW5jZSBwbGFxdWUgY29yZSBieSByZW1vdmluZyBib3VuZGFyeSB2b3hlbHMuCiAgICBJdCB3aWxsIHVuZGVyZXN0aW1hdGUgdm9sdW1lIGJ1dCBjYW4gaGVscCBxdWFudGlmeSBib3VuZGFyeSB1bmNlcnRhaW50eS4KICAgICIiIgogICAgcGxhcXVlID0gbWFzayA9PSBwbGFxdWVfbGFiZWwKICAgIGNvcmUgPSBuZGkuYmluYXJ5X2Vyb3Npb24ocGxhcXVlLCBpdGVyYXRpb25zPWl0ZXJhdGlvbnMpCgogICAgcmVmaW5lZCA9IG1hc2suY29weSgpCiAgICByZWZpbmVkW3BsYXF1ZSAmIH5jb3JlXSA9IDAKCiAgICByZXR1cm4gcmVmaW5lZAoKCmRlZiBsdW1lbl9hZGphY2VudF90cmltKG1hc2ssCiAgICAgICAgICAgICAgICAgICAgICAgIHZlc3NlbF9sYWJlbD0xLAogICAgICAgICAgICAgICAgICAgICAgICBwbGFxdWVfbGFiZWw9MiwKICAgICAgICAgICAgICAgICAgICAgICAgZGlzdGFuY2Vfdm94ZWxzPTEpOgogICAgIiIiCiAgICBSZW1vdmUgcGxhcXVlIHZveGVscyBpbW1lZGlhdGVseSBhZGphY2VudCB0byB0aGUgcHJlZGljdGVkIHZlc3NlbC9sdW1lbiBsYWJlbC4KCiAgICBSYXRpb25hbGU6CiAgICAgICAgSWYgbGFiZWwgMSByZXByZXNlbnRzIG5vcm1hbCB2ZXNzZWwvbHVtZW4gcmVnaW9uIGFuZCBsYWJlbCAyIHBsYXF1ZSwKICAgICAgICBwbGFxdWUgdm94ZWxzIHRvdWNoaW5nIGxhYmVsIDEgbWF5IGJlIHVuY2VydGFpbiBib3VuZGFyeSB2b3hlbHMuCiAgICAiIiIKICAgIHBsYXF1ZSA9IG1hc2sgPT0gcGxhcXVlX2xhYmVsCiAgICB2ZXNzZWwgPSBtYXNrID09IHZlc3NlbF9sYWJlbAoKICAgIHZlc3NlbF9kaWxhdGVkID0gbmRpLmJpbmFyeV9kaWxhdGlvbih2ZXNzZWwsIGl0ZXJhdGlvbnM9ZGlzdGFuY2Vfdm94ZWxzKQogICAgcmVtb3ZlID0gcGxhcXVlICYgdmVzc2VsX2RpbGF0ZWQKCiAgICByZWZpbmVkID0gbWFzay5jb3B5KCkKICAgIHJlZmluZWRbcmVtb3ZlXSA9IDAKCiAgICByZXR1cm4gcmVmaW5lZAoKCmRlZiBpbnRlbnNpdHlfdHJpbSh2b2x1bWUsCiAgICAgICAgICAgICAgICAgICBtYXNrLAogICAgICAgICAgICAgICAgICAgcGxhcXVlX2xhYmVsPTIsCiAgICAgICAgICAgICAgICAgICBoaWdoX2h1X3RocmVzaG9sZD1Ob25lLAogICAgICAgICAgICAgICAgICAgbG93X2h1X3RocmVzaG9sZD1Ob25lKToKICAgICIiIgogICAgUmVtb3ZlIHBsYXF1ZSB2b3hlbHMgb3V0c2lkZSBhIGNvbmZpZ3VyYWJsZSBIVSByYW5nZS4KCiAgICBVc2UgY2F1dGlvdXNseToKICAgICAgICBPbiBjb250cmFzdC1lbmhhbmNlZCBDQ1RBLCBoaWdoIEhVIG1heSBiZSBjYWxjaXVtIE9SIGNvbnRyYXN0LWZpbGxlZCBsdW1lbi4KICAgICAgICBUaGlzIGZ1bmN0aW9uIGlzIHByb3ZpZGVkIGZvciBleHBlcmltZW50cywgbm90IGNsaW5pY2FsIGludGVycHJldGF0aW9uLgogICAgIiIiCiAgICBwbGFxdWUgPSBtYXNrID09IHBsYXF1ZV9sYWJlbAogICAgcmVtb3ZlID0gbnAuemVyb3NfbGlrZShwbGFxdWUsIGR0eXBlPWJvb2wpCgogICAgaWYgaGlnaF9odV90aHJlc2hvbGQgaXMgbm90IE5vbmU6CiAgICAgICAgcmVtb3ZlIHw9IHBsYXF1ZSAmICh2b2x1bWUgPiBoaWdoX2h1X3RocmVzaG9sZCkKCiAgICBpZiBsb3dfaHVfdGhyZXNob2xkIGlzIG5vdCBOb25lOgogICAgICAgIHJlbW92ZSB8PSBwbGFxdWUgJiAodm9sdW1lIDwgbG93X2h1X3RocmVzaG9sZCkKCiAgICByZWZpbmVkID0gbWFzay5jb3B5KCkKICAgIHJlZmluZWRbcmVtb3ZlXSA9IDAKCiAgICByZXR1cm4gcmVmaW5lZAoKCmRlZiByZWZpbmVfcGxhcXVlX21hc2sodm9sdW1lLAogICAgICAgICAgICAgICAgICAgICAgIG1hc2ssCiAgICAgICAgICAgICAgICAgICAgICAgc3BhY2luZywKICAgICAgICAgICAgICAgICAgICAgICByZW1vdmVfc21hbGw9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICBtaW5fY29tcG9uZW50X3ZveGVscz0xMCwKICAgICAgICAgICAgICAgICAgICAgICB0cmltX2x1bWVuX2FkamFjZW50PVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgbHVtZW5fZGlzdGFuY2Vfdm94ZWxzPTEsCiAgICAgICAgICAgICAgICAgICAgICAgZXJvZGVfY29yZT1GYWxzZSwKICAgICAgICAgICAgICAgICAgICAgICBlcm9zaW9uX2l0ZXJhdGlvbnM9MSwKICAgICAgICAgICAgICAgICAgICAgICBoaWdoX2h1X3RocmVzaG9sZD1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgIGxvd19odV90aHJlc2hvbGQ9Tm9uZSk6CiAgICAiIiIKICAgIEFwcGx5IGEgY29uZmlndXJhYmxlIHNlcXVlbmNlIG9mIGV4cGVyaW1lbnRhbCByZWZpbmVtZW50IHN0ZXBzLgoKICAgIFJlY29tbWVuZGVkIHN0YXJ0aW5nIHBvaW50OgogICAgICAgIHJlbW92ZV9zbWFsbD1UcnVlCiAgICAgICAgdHJpbV9sdW1lbl9hZGphY2VudD1UcnVlCiAgICAgICAgZXJvZGVfY29yZT1GYWxzZQogICAgICAgIGhpZ2hfaHVfdGhyZXNob2xkPU5vbmUKCiAgICBUaGlzIHJldHVybnMgYSBSZWZpbmVtZW50UmVzdWx0IG9iamVjdCBhbmQgcHJlc2VydmVzIG5vbi1wbGFxdWUgbGFiZWxzLgogICAgIiIiCgogICAgb3JpZ2luYWwgPSBtYXNrLmNvcHkoKQogICAgcmVmaW5lZCA9IG1hc2suY29weSgpCgogICAgcGFyYW1zID0gewogICAgICAgICJyZW1vdmVfc21hbGwiOiByZW1vdmVfc21hbGwsCiAgICAgICAgIm1pbl9jb21wb25lbnRfdm94ZWxzIjogbWluX2NvbXBvbmVudF92b3hlbHMsCiAgICAgICAgInRyaW1fbHVtZW5fYWRqYWNlbnQiOiB0cmltX2x1bWVuX2FkamFjZW50LAogICAgICAgICJsdW1lbl9kaXN0YW5jZV92b3hlbHMiOiBsdW1lbl9kaXN0YW5jZV92b3hlbHMsCiAgICAgICAgImVyb2RlX2NvcmUiOiBlcm9kZV9jb3JlLAogICAgICAgICJlcm9zaW9uX2l0ZXJhdGlvbnMiOiBlcm9zaW9uX2l0ZXJhdGlvbnMsCiAgICAgICAgImhpZ2hfaHVfdGhyZXNob2xkIjogaGlnaF9odV90aHJlc2hvbGQsCiAgICAgICAgImxvd19odV90aHJlc2hvbGQiOiBsb3dfaHVfdGhyZXNob2xkLAogICAgfQoKICAgIGlmIHJlbW92ZV9zbWFsbDoKICAgICAgICByZWZpbmVkID0gcmVtb3ZlX3NtYWxsX2NvbXBvbmVudHMoCiAgICAgICAgICAgIHJlZmluZWQsCiAgICAgICAgICAgIG1pbl92b3hlbHM9bWluX2NvbXBvbmVudF92b3hlbHMsCiAgICAgICAgICAgIHBsYXF1ZV9sYWJlbD0yLAogICAgICAgICkKCiAgICBpZiB0cmltX2x1bWVuX2FkamFjZW50OgogICAgICAgIHJlZmluZWQgPSBsdW1lbl9hZGphY2VudF90cmltKAogICAgICAgICAgICByZWZpbmVkLAogICAgICAgICAgICB2ZXNzZWxfbGFiZWw9MSwKICAgICAgICAgICAgcGxhcXVlX2xhYmVsPTIsCiAgICAgICAgICAgIGRpc3RhbmNlX3ZveGVscz1sdW1lbl9kaXN0YW5jZV92b3hlbHMsCiAgICAgICAgKQoKICAgIGlmIGhpZ2hfaHVfdGhyZXNob2xkIGlzIG5vdCBOb25lIG9yIGxvd19odV90aHJlc2hvbGQgaXMgbm90IE5vbmU6CiAgICAgICAgcmVmaW5lZCA9IGludGVuc2l0eV90cmltKAogICAgICAgICAgICB2b2x1bWUsCiAgICAgICAgICAgIHJlZmluZWQsCiAgICAgICAgICAgIHBsYXF1ZV9sYWJlbD0yLAogICAgICAgICAgICBoaWdoX2h1X3RocmVzaG9sZD1oaWdoX2h1X3RocmVzaG9sZCwKICAgICAgICAgICAgbG93X2h1X3RocmVzaG9sZD1sb3dfaHVfdGhyZXNob2xkLAogICAgICAgICkKCiAgICBpZiBlcm9kZV9jb3JlOgogICAgICAgIHJlZmluZWQgPSBlcm9kZV9wbGFxdWVfYm91bmRhcnkoCiAgICAgICAgICAgIHJlZmluZWQsCiAgICAgICAgICAgIGl0ZXJhdGlvbnM9ZXJvc2lvbl9pdGVyYXRpb25zLAogICAgICAgICAgICBwbGFxdWVfbGFiZWw9MiwKICAgICAgICApCgogICAgcmVtb3ZlZCA9IChvcmlnaW5hbCA9PSAyKSAmIChyZWZpbmVkICE9IDIpCgogICAgcmV0dXJuIFJlZmluZW1lbnRSZXN1bHQoCiAgICAgICAgb3JpZ2luYWxfbWFzaz1vcmlnaW5hbCwKICAgICAgICByZWZpbmVkX21hc2s9cmVmaW5lZCwKICAgICAgICByZW1vdmVkX21hc2s9cmVtb3ZlZCwKICAgICAgICBzcGFjaW5nPXNwYWNpbmcsCiAgICAgICAgbWV0aG9kPSJyZW1vdmVfc21hbGwgKyBsdW1lbl9hZGphY2VudCArIG9wdGlvbmFsX2ludGVuc2l0eSArIG9wdGlvbmFsX2Vyb3Npb24iLAogICAgICAgIHBhcmFtZXRlcnM9cGFyYW1zLAogICAgKQo=',
    'report.py': 'IiIiSFRNTCByZXBvcnQgZ2VuZXJhdGlvbiBmb3IgT3BlblBsYXF1ZSBUUFYgd29ya2Zsb3dzLiIiIgoKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gaHRtbCBpbXBvcnQgZXNjYXBlCmZyb20gdHlwaW5nIGltcG9ydCBJdGVyYWJsZSwgTWFwcGluZywgT3B0aW9uYWwKaW1wb3J0IGJhc2U2NAppbXBvcnQgbnVtcHkgYXMgbnAKCgpkZWYgX21hc2tfcG9zaXRpdmUobWFzaywgbGFiZWw9Mik6CiAgICBhcnIgPSBucC5hc2FycmF5KG1hc2spCiAgICBpZiBsYWJlbCBpcyBOb25lIG9yIChhcnIuc2l6ZSBhbmQgbnAubmFubWF4KGFycikgPD0gMSk6CiAgICAgICAgcmV0dXJuIGFyciA+IDAKICAgIHJldHVybiBhcnIgPT0gbGFiZWwKCgpkZWYgX2Jlc3Rfc2xpY2UobWFzaywgbGFiZWw9Mik6CiAgICBwb3MgPSBfbWFza19wb3NpdGl2ZShtYXNrLCBsYWJlbD1sYWJlbCkKICAgIGNvdW50cyA9IG5wLnN1bShwb3MsIGF4aXM9KDEsIDIpKQogICAgcmV0dXJuIGludChucC5hcmdtYXgoY291bnRzKSkgaWYgY291bnRzLnNpemUgZWxzZSAwCgoKZGVmIHNhdmVfb3ZlcmxheV9wbmcodm9sdW1lLCBtYXNrLCBwYXRoLCB0aXRsZSwgej1Ob25lLCBsYWJlbD0yLCB2bWluPS0yMDAsIHZtYXg9ODAwKToKICAgIGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgIGlmIHogaXMgTm9uZToKICAgICAgICB6ID0gX2Jlc3Rfc2xpY2UobWFzaywgbGFiZWw9bGFiZWwpCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcGx0LmZpZ3VyZShmaWdzaXplPSg2LCA2KSkKICAgIHBsdC5pbXNob3codm9sdW1lW3pdLCBjbWFwPSJncmF5Iiwgdm1pbj12bWluLCB2bWF4PXZtYXgpCiAgICBwbHQuaW1zaG93KF9tYXNrX3Bvc2l0aXZlKG1hc2ssIGxhYmVsPWxhYmVsKVt6XSwgYWxwaGE9MC40NSkKICAgIHBsdC50aXRsZShmInt0aXRsZX0g4oCUIHNsaWNlIHt6fSIpCiAgICBwbHQuYXhpcygib2ZmIikKICAgIHBsdC50aWdodF9sYXlvdXQoKQogICAgcGx0LnNhdmVmaWcocGF0aCwgZHBpPTE1MCkKICAgIHBsdC5jbG9zZSgpCiAgICByZXR1cm4gcGF0aAoKCmRlZiBfaW1nX2RhdGFfdXJpKHBhdGgpOgogICAgZGF0YSA9IFBhdGgocGF0aCkucmVhZF9ieXRlcygpCiAgICByZXR1cm4gImRhdGE6aW1hZ2UvcG5nO2Jhc2U2NCwiICsgYmFzZTY0LmI2NGVuY29kZShkYXRhKS5kZWNvZGUoImFzY2lpIikKCgpkZWYgX2ZtdCh4KToKICAgIHJldHVybiBmIntmbG9hdCh4KTouMmZ9IgoKCmRlZiB3cml0ZV9odG1sX3JlcG9ydCgKICAgIG91dHB1dF9odG1sLAogICAgcmVwb3J0czogSXRlcmFibGUsCiAgICByZWZpbmVkX3Jlc3VsdHM6IE1hcHBpbmdbc3RyLCBvYmplY3RdLAogICAgY29yZV9yZXN1bHRzOiBNYXBwaW5nW3N0ciwgb2JqZWN0XSwKICAgIHVuY2VydGFpbnR5X3N1bW1hcnksCiAgICBvdmVybGF5X2RpcjogT3B0aW9uYWxbc3RyXSA9IE5vbmUsCiAgICB0aXRsZTogc3RyID0gIk9wZW5QbGFxdWUgVFBWIEJvdW5kYXJ5IFJlZmluZW1lbnQgUmVwb3J0IiwKKToKICAgICIiIldyaXRlIGEgc2VsZi1jb250YWluZWQgSFRNTCByZXBvcnQgd2l0aCBUUFYgdGFibGVzIGFuZCBvdmVybGF5IFBOR3MuIiIiCiAgICBvdXRwdXRfaHRtbCA9IFBhdGgob3V0cHV0X2h0bWwpCiAgICBvdmVybGF5X2RpciA9IFBhdGgob3ZlcmxheV9kaXIpIGlmIG92ZXJsYXlfZGlyIGVsc2Ugb3V0cHV0X2h0bWwud2l0aF9zdWZmaXgoIiIpLnBhcmVudCAvICJvcGVucGxhcXVlX3JlcG9ydF9hc3NldHMiCiAgICBvdmVybGF5X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgcmVwb3J0X2xpc3QgPSBsaXN0KHJlcG9ydHMpCiAgICBvdmVybGF5X3BhdGhzID0ge30KICAgIGZvciByZXBvcnQgaW4gcmVwb3J0X2xpc3Q6CiAgICAgICAgbmFtZSA9IHJlcG9ydC5uYW1lCiAgICAgICAgcmF3X3BhdGggPSBzYXZlX292ZXJsYXlfcG5nKHJlcG9ydC52b2x1bWUsIHJlcG9ydC5tYXNrLCBvdmVybGF5X2RpciAvIGYie25hbWV9X3Jhd19vdmVybGF5LnBuZyIsIGYie25hbWV9IHJhdyBwbGFxdWUiKQogICAgICAgIHJlZmluZWRfcGF0aCA9IHNhdmVfb3ZlcmxheV9wbmcocmVwb3J0LnZvbHVtZSwgcmVmaW5lZF9yZXN1bHRzW25hbWVdLnJlZmluZWRfbWFzaywgb3ZlcmxheV9kaXIgLyBmIntuYW1lfV9yZWZpbmVkX292ZXJsYXkucG5nIiwgZiJ7bmFtZX0gcmVmaW5lZCBwbGFxdWUiKQogICAgICAgIGNvcmVfcGF0aCA9IHNhdmVfb3ZlcmxheV9wbmcocmVwb3J0LnZvbHVtZSwgY29yZV9yZXN1bHRzW25hbWVdLnJlZmluZWRfbWFzaywgb3ZlcmxheV9kaXIgLyBmIntuYW1lfV9jb3JlX292ZXJsYXkucG5nIiwgZiJ7bmFtZX0gaGlnaC1jb25maWRlbmNlIGNvcmUiKQogICAgICAgIG92ZXJsYXlfcGF0aHNbbmFtZV0gPSAocmF3X3BhdGgsIHJlZmluZWRfcGF0aCwgY29yZV9wYXRoKQoKICAgIHJvd3NfaHRtbCA9IFtdCiAgICBmb3Igcm93IGluIHVuY2VydGFpbnR5X3N1bW1hcnkucm93cygpOgogICAgICAgIHJvd3NfaHRtbC5hcHBlbmQoCiAgICAgICAgICAgICI8dHI+IgogICAgICAgICAgICBmIjx0ZD57ZXNjYXBlKHJvd1sndmVzc2VsJ10pfTwvdGQ+IgogICAgICAgICAgICBmIjx0ZD57X2ZtdChyb3dbJ2NvcmVfdHB2X21tMyddKX08L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e19mbXQocm93WydyZWZpbmVkX3Rwdl9tbTMnXSl9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntfZm10KHJvd1sncmF3X3Rwdl9tbTMnXSl9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntfZm10KHJvd1sncmVtb3ZlZF9ib3VuZGFyeV9tbTMnXSl9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntfZm10KHJvd1snaW50ZXJ2YWxfbG93X21tMyddKX0g4oCTIHtfZm10KHJvd1snaW50ZXJ2YWxfaGlnaF9tbTMnXSl9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntfZm10KHJvd1sndW5jZXJ0YWludHlfd2lkdGhfbW0zJ10pfTwvdGQ+IgogICAgICAgICAgICAiPC90cj4iCiAgICAgICAgKQoKICAgIG92ZXJsYXlfaHRtbCA9IFtdCiAgICBmb3IgcmVwb3J0IGluIHJlcG9ydF9saXN0OgogICAgICAgIG5hbWUgPSByZXBvcnQubmFtZQogICAgICAgIHJhdywgcmVmaW5lZCwgY29yZSA9IG92ZXJsYXlfcGF0aHNbbmFtZV0KICAgICAgICBvdmVybGF5X2h0bWwuYXBwZW5kKGYiPGgyPntlc2NhcGUobmFtZSl9IG92ZXJsYXlzPC9oMj48ZGl2IGNsYXNzPSdncmlkJz4iKQogICAgICAgIGZvciBsYWJlbCwgcGF0aCBpbiAoKCJSYXciLCByYXcpLCAoIlJlZmluZWQiLCByZWZpbmVkKSwgKCJDb3JlIiwgY29yZSkpOgogICAgICAgICAgICBvdmVybGF5X2h0bWwuYXBwZW5kKAogICAgICAgICAgICAgICAgZiI8ZmlndXJlPjxpbWcgc3JjPSd7X2ltZ19kYXRhX3VyaShwYXRoKX0nIGFsdD0ne2VzY2FwZShuYW1lKX0ge2xhYmVsfSBvdmVybGF5Jz4iCiAgICAgICAgICAgICAgICBmIjxmaWdjYXB0aW9uPntlc2NhcGUobmFtZSl9IHtsYWJlbH08L2ZpZ2NhcHRpb24+PC9maWd1cmU+IgogICAgICAgICAgICApCiAgICAgICAgb3ZlcmxheV9odG1sLmFwcGVuZCgiPC9kaXY+IikKCiAgICBodG1sID0gZiIiIjwhZG9jdHlwZSBodG1sPgo8aHRtbCBsYW5nPSJlbiI+CjxoZWFkPgo8bWV0YSBjaGFyc2V0PSJ1dGYtOCI+Cjx0aXRsZT57ZXNjYXBlKHRpdGxlKX08L3RpdGxlPgo8c3R5bGU+CmJvZHkge3sgZm9udC1mYW1pbHk6IC1hcHBsZS1zeXN0ZW0sIEJsaW5rTWFjU3lzdGVtRm9udCwgU2Vnb2UgVUksIHNhbnMtc2VyaWY7IG1hcmdpbjogMzJweDsgY29sb3I6ICMyMjI7IH19CmgxIHt7IG1hcmdpbi1ib3R0b206IDAuMjVyZW07IH19Ci5ub3RpY2Uge3sgcGFkZGluZzogMTJweCAxNHB4OyBiYWNrZ3JvdW5kOiAjZmZmM2NkOyBib3JkZXI6IDFweCBzb2xpZCAjZmZlMDhhOyBib3JkZXItcmFkaXVzOiA4cHg7IH19CnRhYmxlIHt7IGJvcmRlci1jb2xsYXBzZTogY29sbGFwc2U7IHdpZHRoOiAxMDAlOyBtYXJnaW46IDIwcHggMCAzMHB4OyB9fQp0aCwgdGQge3sgYm9yZGVyOiAxcHggc29saWQgI2RkZDsgcGFkZGluZzogOHB4IDEwcHg7IHRleHQtYWxpZ246IHJpZ2h0OyB9fQp0aDpmaXJzdC1jaGlsZCwgdGQ6Zmlyc3QtY2hpbGQge3sgdGV4dC1hbGlnbjogbGVmdDsgfX0KdGgge3sgYmFja2dyb3VuZDogI2Y2ZjZmNjsgfX0KLmdyaWQge3sgZGlzcGxheTogZ3JpZDsgZ3JpZC10ZW1wbGF0ZS1jb2x1bW5zOiByZXBlYXQoYXV0by1maXQsIG1pbm1heCgyNDBweCwgMWZyKSk7IGdhcDogMThweDsgfX0KZmlndXJlIHt7IG1hcmdpbjogMDsgfX0KaW1nIHt7IHdpZHRoOiAxMDAlOyBib3JkZXI6IDFweCBzb2xpZCAjZGRkOyBib3JkZXItcmFkaXVzOiA4cHg7IH19CmZpZ2NhcHRpb24ge3sgdGV4dC1hbGlnbjogY2VudGVyOyBmb250LXNpemU6IDAuOXJlbTsgY29sb3I6ICM1NTU7IG1hcmdpbi10b3A6IDZweDsgfX0KLnNtYWxsIHt7IGNvbG9yOiAjNjY2OyBmb250LXNpemU6IDAuOTJyZW07IH19Cjwvc3R5bGU+CjwvaGVhZD4KPGJvZHk+CjxoMT57ZXNjYXBlKHRpdGxlKX08L2gxPgo8cCBjbGFzcz0ibm90aWNlIj48c3Ryb25nPlJlc2VhcmNoIHVzZSBvbmx5Ljwvc3Ryb25nPiBUaGlzIHJlcG9ydCBpcyBub3QgY2xpbmljYWxseSB2YWxpZGF0ZWQgYW5kIGlzIG5vdCBmb3IgZGlhZ25vc2lzIG9yIG1lZGljYWwgZGVjaXNpb24tbWFraW5nLjwvcD4KPHAgY2xhc3M9InNtYWxsIj5VbmNlcnRhaW50eSBpbnRlcnZhbCBpcyByZXBvcnRlZCBhcyBoaWdoLWNvbmZpZGVuY2UgY29yZSBUUFYgdG8gcmF3IEFJIFRQViwgd2l0aCByZWZpbmVkIFRQViBhcyB0aGUgbWlkcG9pbnQtc3R5bGUgZXN0aW1hdGUuPC9wPgo8aDI+VFBWIHVuY2VydGFpbnR5IHN1bW1hcnk8L2gyPgo8dGFibGU+Cjx0aGVhZD48dHI+PHRoPlZlc3NlbDwvdGg+PHRoPkNvcmUgVFBWIG1twrM8L3RoPjx0aD5SZWZpbmVkIFRQViBtbcKzPC90aD48dGg+UmF3IFRQViBtbcKzPC90aD48dGg+UmVtb3ZlZCBib3VuZGFyeSBtbcKzPC90aD48dGg+SW50ZXJ2YWwgbW3CszwvdGg+PHRoPldpZHRoIG1twrM8L3RoPjwvdHI+PC90aGVhZD4KPHRib2R5Pgp7Jycuam9pbihyb3dzX2h0bWwpfQo8L3Rib2R5Pgo8L3RhYmxlPgp7Jycuam9pbihvdmVybGF5X2h0bWwpfQo8L2JvZHk+CjwvaHRtbD4KIiIiCiAgICBvdXRwdXRfaHRtbC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgb3V0cHV0X2h0bWwud3JpdGVfdGV4dChodG1sLCBlbmNvZGluZz0idXRmLTgiKQogICAgcmV0dXJuIG91dHB1dF9odG1sCg==',
    'uncertainty.py': 'IiIiVFBWIHVuY2VydGFpbnR5IHN1bW1hcmllcyBmb3IgT3BlblBsYXF1ZS4iIiIKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgYXNkaWN0CmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBJdGVyYWJsZSwgTWFwcGluZwoKCkBkYXRhY2xhc3MKY2xhc3MgVmVzc2VsVFBWVW5jZXJ0YWludHk6CiAgICB2ZXNzZWw6IHN0cgogICAgY29yZV90cHZfbW0zOiBmbG9hdAogICAgcmVmaW5lZF90cHZfbW0zOiBmbG9hdAogICAgcmF3X3Rwdl9tbTM6IGZsb2F0CiAgICByZW1vdmVkX2JvdW5kYXJ5X21tMzogZmxvYXQKCiAgICBAcHJvcGVydHkKICAgIGRlZiBpbnRlcnZhbF9sb3dfbW0zKHNlbGYpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBzZWxmLmNvcmVfdHB2X21tMwoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGludGVydmFsX2hpZ2hfbW0zKHNlbGYpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBzZWxmLnJhd190cHZfbW0zCgogICAgQHByb3BlcnR5CiAgICBkZWYgaW50ZXJ2YWxfbWlkX21tMyhzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gc2VsZi5yZWZpbmVkX3Rwdl9tbTMKCiAgICBAcHJvcGVydHkKICAgIGRlZiB1bmNlcnRhaW50eV93aWR0aF9tbTMoc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIHNlbGYuaW50ZXJ2YWxfaGlnaF9tbTMgLSBzZWxmLmludGVydmFsX2xvd19tbTMKCiAgICBkZWYgdG9fZGljdChzZWxmKToKICAgICAgICBkID0gYXNkaWN0KHNlbGYpCiAgICAgICAgZC51cGRhdGUoewogICAgICAgICAgICAiaW50ZXJ2YWxfbG93X21tMyI6IHNlbGYuaW50ZXJ2YWxfbG93X21tMywKICAgICAgICAgICAgImludGVydmFsX21pZF9tbTMiOiBzZWxmLmludGVydmFsX21pZF9tbTMsCiAgICAgICAgICAgICJpbnRlcnZhbF9oaWdoX21tMyI6IHNlbGYuaW50ZXJ2YWxfaGlnaF9tbTMsCiAgICAgICAgICAgICJ1bmNlcnRhaW50eV93aWR0aF9tbTMiOiBzZWxmLnVuY2VydGFpbnR5X3dpZHRoX21tMywKICAgICAgICB9KQogICAgICAgIHJldHVybiBkCgoKQGRhdGFjbGFzcwpjbGFzcyBUUFZVbmNlcnRhaW50eVN1bW1hcnk6CiAgICB2ZXNzZWxzOiBEaWN0W3N0ciwgVmVzc2VsVFBWVW5jZXJ0YWludHldCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG90YWxfY29yZV90cHZfbW0zKHNlbGYpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBzdW0odi5jb3JlX3Rwdl9tbTMgZm9yIHYgaW4gc2VsZi52ZXNzZWxzLnZhbHVlcygpKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRvdGFsX3JlZmluZWRfdHB2X21tMyhzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gc3VtKHYucmVmaW5lZF90cHZfbW0zIGZvciB2IGluIHNlbGYudmVzc2Vscy52YWx1ZXMoKSkKCiAgICBAcHJvcGVydHkKICAgIGRlZiB0b3RhbF9yYXdfdHB2X21tMyhzZWxmKSAtPiBmbG9hdDoKICAgICAgICByZXR1cm4gc3VtKHYucmF3X3Rwdl9tbTMgZm9yIHYgaW4gc2VsZi52ZXNzZWxzLnZhbHVlcygpKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRvdGFsX3JlbW92ZWRfYm91bmRhcnlfbW0zKHNlbGYpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBzdW0odi5yZW1vdmVkX2JvdW5kYXJ5X21tMyBmb3IgdiBpbiBzZWxmLnZlc3NlbHMudmFsdWVzKCkpCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG90YWxfdW5jZXJ0YWludHlfd2lkdGhfbW0zKHNlbGYpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBzZWxmLnRvdGFsX3Jhd190cHZfbW0zIC0gc2VsZi50b3RhbF9jb3JlX3Rwdl9tbTMKCiAgICBkZWYgdG90YWxfcm93KHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInZlc3NlbCI6ICJUT1RBTCIsCiAgICAgICAgICAgICJjb3JlX3Rwdl9tbTMiOiBzZWxmLnRvdGFsX2NvcmVfdHB2X21tMywKICAgICAgICAgICAgInJlZmluZWRfdHB2X21tMyI6IHNlbGYudG90YWxfcmVmaW5lZF90cHZfbW0zLAogICAgICAgICAgICAicmF3X3Rwdl9tbTMiOiBzZWxmLnRvdGFsX3Jhd190cHZfbW0zLAogICAgICAgICAgICAicmVtb3ZlZF9ib3VuZGFyeV9tbTMiOiBzZWxmLnRvdGFsX3JlbW92ZWRfYm91bmRhcnlfbW0zLAogICAgICAgICAgICAiaW50ZXJ2YWxfbG93X21tMyI6IHNlbGYudG90YWxfY29yZV90cHZfbW0zLAogICAgICAgICAgICAiaW50ZXJ2YWxfbWlkX21tMyI6IHNlbGYudG90YWxfcmVmaW5lZF90cHZfbW0zLAogICAgICAgICAgICAiaW50ZXJ2YWxfaGlnaF9tbTMiOiBzZWxmLnRvdGFsX3Jhd190cHZfbW0zLAogICAgICAgICAgICAidW5jZXJ0YWludHlfd2lkdGhfbW0zIjogc2VsZi50b3RhbF91bmNlcnRhaW50eV93aWR0aF9tbTMsCiAgICAgICAgfQoKICAgIGRlZiByb3dzKHNlbGYpOgogICAgICAgIHJldHVybiBbdi50b19kaWN0KCkgZm9yIHYgaW4gc2VsZi52ZXNzZWxzLnZhbHVlcygpXSArIFtzZWxmLnRvdGFsX3JvdygpXQoKCmRlZiBtYWtlX3Rwdl91bmNlcnRhaW50eV9zdW1tYXJ5KHJhd19yZXBvcnRzOiBJdGVyYWJsZSwgcmVmaW5lZF9yZXN1bHRzOiBNYXBwaW5nW3N0ciwgb2JqZWN0XSwgY29yZV9yZXN1bHRzOiBNYXBwaW5nW3N0ciwgb2JqZWN0XSkgLT4gVFBWVW5jZXJ0YWludHlTdW1tYXJ5OgogICAgdmVzc2VscyA9IHt9CiAgICBmb3IgcmVwb3J0IGluIHJhd19yZXBvcnRzOgogICAgICAgIG5hbWUgPSByZXBvcnQubmFtZQogICAgICAgIHZlc3NlbHNbbmFtZV0gPSBWZXNzZWxUUFZVbmNlcnRhaW50eSgKICAgICAgICAgICAgdmVzc2VsPW5hbWUsCiAgICAgICAgICAgIGNvcmVfdHB2X21tMz1mbG9hdChjb3JlX3Jlc3VsdHNbbmFtZV0ucmVmaW5lZF90cHZfbW0zKSwKICAgICAgICAgICAgcmVmaW5lZF90cHZfbW0zPWZsb2F0KHJlZmluZWRfcmVzdWx0c1tuYW1lXS5yZWZpbmVkX3Rwdl9tbTMpLAogICAgICAgICAgICByYXdfdHB2X21tMz1mbG9hdChyZXBvcnQudHB2X21tMyksCiAgICAgICAgICAgIHJlbW92ZWRfYm91bmRhcnlfbW0zPWZsb2F0KHJlZmluZWRfcmVzdWx0c1tuYW1lXS5yZW1vdmVkX3ZvbHVtZV9tbTMpLAogICAgICAgICkKICAgIHJldHVybiBUUFZVbmNlcnRhaW50eVN1bW1hcnkodmVzc2Vscz12ZXNzZWxzKQo=',
    'tuning.py': 'IiIiQm91bmRhcnktcmVmaW5lbWVudCBwYXJhbWV0ZXIgdHVuaW5nIGZvciBPcGVuUGxhcXVlLgoKVGhpcyBtb2R1bGUgcGVyZm9ybXMgdW5zdXBlcnZpc2VkL3Nhbml0eS1jaGVjayB0dW5pbmcgb3ZlciB0aGUgQUkgcGxhcXVlIG1hc2tzLgpJdCBkb2VzIG5vdCByZXF1aXJlIGEgbWFudWFsIGdyb3VuZC10cnV0aCBhbm5vdGF0aW9uLiBUaGUgZGVmYXVsdCBvYmplY3RpdmUgZmF2b3JzCnN0YWJsZSwgY29uc2VydmF0aXZlIHJlZmluZW1lbnRzIHRoYXQgcmVtb3ZlIGxpa2VseSBib3VuZGFyeS9ub2lzZSB2b3hlbHMgd2l0aG91dApjb2xsYXBzaW5nIHRoZSBwcmVkaWN0ZWQgcGxhcXVlIHZvbHVtZS4KClJlc2VhcmNoIHVzZSBvbmx5LiBOb3QgY2xpbmljYWxseSB2YWxpZGF0ZWQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QKZnJvbSBpdGVydG9vbHMgaW1wb3J0IHByb2R1Y3QKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBEaWN0LCBJdGVyYWJsZSwgTWFwcGluZywgT3B0aW9uYWwsIFNlcXVlbmNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNjaXB5IGltcG9ydCBuZGltYWdlIGFzIG5kaQoKZnJvbSAuYm91bmRhcnkgaW1wb3J0IHJlZmluZV9wbGFxdWVfbWFzawoKCkRFRkFVTFRfUEFSQU1FVEVSX0dSSUQgPSB7CiAgICAibWluX2NvbXBvbmVudF92b3hlbHMiOiBbMSwgNSwgMTAsIDI1LCA1MF0sCiAgICAibHVtZW5fZGlzdGFuY2Vfdm94ZWxzIjogWzAsIDEsIDJdLAogICAgImhpZ2hfaHVfdGhyZXNob2xkIjogW05vbmUsIDcwMCwgODUwLCAxMDAwXSwKICAgICJsb3dfaHVfdGhyZXNob2xkIjogW05vbmUsIC0xMDAsIC01MF0sCiAgICAiZXJvZGVfY29yZSI6IFtGYWxzZSwgVHJ1ZV0sCiAgICAiZXJvc2lvbl9pdGVyYXRpb25zIjogWzFdLAp9CgoKQGRhdGFjbGFzcwpjbGFzcyBUdW5pbmdSb3c6CiAgICB2ZXNzZWw6IHN0cgogICAgY2FuZGlkYXRlX2lkOiBpbnQKICAgIG1pbl9jb21wb25lbnRfdm94ZWxzOiBpbnQKICAgIGx1bWVuX2Rpc3RhbmNlX3ZveGVsczogaW50CiAgICBoaWdoX2h1X3RocmVzaG9sZDogT3B0aW9uYWxbZmxvYXRdCiAgICBsb3dfaHVfdGhyZXNob2xkOiBPcHRpb25hbFtmbG9hdF0KICAgIGVyb2RlX2NvcmU6IGJvb2wKICAgIGVyb3Npb25faXRlcmF0aW9uczogaW50CiAgICByYXdfdHB2X21tMzogZmxvYXQKICAgIHJlZmluZWRfdHB2X21tMzogZmxvYXQKICAgIHJlbW92ZWRfbW0zOiBmbG9hdAogICAgcmV0YWluZWRfZnJhY3Rpb246IGZsb2F0CiAgICByZW1vdmVkX2ZyYWN0aW9uOiBmbG9hdAogICAgY29tcG9uZW50c19iZWZvcmU6IGludAogICAgY29tcG9uZW50c19hZnRlcjogaW50CiAgICBsYXJnZXN0X2NvbXBvbmVudF9mcmFjdGlvbjogZmxvYXQKICAgIG1lYW5faHU6IE9wdGlvbmFsW2Zsb2F0XQogICAgbWVkaWFuX2h1OiBPcHRpb25hbFtmbG9hdF0KICAgIHNjb3JlOiBmbG9hdAogICAgcmVqZWN0X3JlYXNvbjogc3RyID0gIiIKCiAgICBkZWYgdG9fZGljdChzZWxmKSAtPiBkaWN0OgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpAZGF0YWNsYXNzCmNsYXNzIFR1bmluZ1Jlc3VsdDoKICAgIHJvd3M6IGxpc3RbVHVuaW5nUm93XQogICAgYmVzdF9ieV92ZXNzZWw6IERpY3Rbc3RyLCBUdW5pbmdSb3ddCiAgICBzZWxlY3RlZF9wYXJhbXM6IGRpY3QKICAgIG5vdGVzOiBzdHIKCiAgICBkZWYgdG9fcm93cyhzZWxmKSAtPiBsaXN0W2RpY3RdOgogICAgICAgIHJldHVybiBbci50b19kaWN0KCkgZm9yIHIgaW4gc2VsZi5yb3dzXQoKICAgIGRlZiBzYXZlX2NzdihzZWxmLCBwYXRoKSAtPiBQYXRoOgogICAgICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICAgICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIHJvd3MgPSBzZWxmLnRvX3Jvd3MoKQogICAgICAgIGlmIG5vdCByb3dzOgogICAgICAgICAgICBwYXRoLndyaXRlX3RleHQoIiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgICAgIHJldHVybiBwYXRoCiAgICAgICAgd2l0aCBwYXRoLm9wZW4oInciLCBuZXdsaW5lPSIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihmLCBmaWVsZG5hbWVzPWxpc3Qocm93c1swXS5rZXlzKCkpKQogICAgICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgICAgICB3cml0ZXIud3JpdGVyb3dzKHJvd3MpCiAgICAgICAgcmV0dXJuIHBhdGgKCiAgICBkZWYgc2F2ZV9qc29uKHNlbGYsIHBhdGgpIC0+IFBhdGg6CiAgICAgICAgcGF0aCA9IFBhdGgocGF0aCkKICAgICAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgcGF5bG9hZCA9IHsKICAgICAgICAgICAgInNlbGVjdGVkX3BhcmFtcyI6IHNlbGYuc2VsZWN0ZWRfcGFyYW1zLAogICAgICAgICAgICAiYmVzdF9ieV92ZXNzZWwiOiB7azogdi50b19kaWN0KCkgZm9yIGssIHYgaW4gc2VsZi5iZXN0X2J5X3Zlc3NlbC5pdGVtcygpfSwKICAgICAgICAgICAgInJvd3MiOiBzZWxmLnRvX3Jvd3MoKSwKICAgICAgICAgICAgIm5vdGVzIjogc2VsZi5ub3RlcywKICAgICAgICB9CiAgICAgICAgcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMocGF5bG9hZCwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgIHJldHVybiBwYXRoCgoKZGVmIF9wYXJhbWV0ZXJfY2FuZGlkYXRlcyhncmlkOiBPcHRpb25hbFtNYXBwaW5nW3N0ciwgU2VxdWVuY2VdXSA9IE5vbmUpOgogICAgZ3JpZCA9IGRpY3QoREVGQVVMVF9QQVJBTUVURVJfR1JJRCBpZiBncmlkIGlzIE5vbmUgZWxzZSBncmlkKQogICAga2V5cyA9IGxpc3QoZ3JpZC5rZXlzKCkpCiAgICBmb3IgaSwgdmFsdWVzIGluIGVudW1lcmF0ZShwcm9kdWN0KCpbZ3JpZFtrXSBmb3IgayBpbiBrZXlzXSkpOgogICAgICAgIHAgPSBkaWN0KHppcChrZXlzLCB2YWx1ZXMpKQogICAgICAgIGlmIHAuZ2V0KCJsdW1lbl9kaXN0YW5jZV92b3hlbHMiLCAxKSA9PSAwOgogICAgICAgICAgICB0cmltX2x1bWVuX2FkamFjZW50ID0gRmFsc2UKICAgICAgICBlbHNlOgogICAgICAgICAgICB0cmltX2x1bWVuX2FkamFjZW50ID0gVHJ1ZQogICAgICAgIHlpZWxkIGksIHsKICAgICAgICAgICAgInJlbW92ZV9zbWFsbCI6IFRydWUsCiAgICAgICAgICAgICJtaW5fY29tcG9uZW50X3ZveGVscyI6IGludChwLmdldCgibWluX2NvbXBvbmVudF92b3hlbHMiLCAxMCkpLAogICAgICAgICAgICAidHJpbV9sdW1lbl9hZGphY2VudCI6IHRyaW1fbHVtZW5fYWRqYWNlbnQsCiAgICAgICAgICAgICJsdW1lbl9kaXN0YW5jZV92b3hlbHMiOiBpbnQocC5nZXQoImx1bWVuX2Rpc3RhbmNlX3ZveGVscyIsIDEpKSwKICAgICAgICAgICAgImVyb2RlX2NvcmUiOiBib29sKHAuZ2V0KCJlcm9kZV9jb3JlIiwgRmFsc2UpKSwKICAgICAgICAgICAgImVyb3Npb25faXRlcmF0aW9ucyI6IGludChwLmdldCgiZXJvc2lvbl9pdGVyYXRpb25zIiwgMSkpLAogICAgICAgICAgICAiaGlnaF9odV90aHJlc2hvbGQiOiBwLmdldCgiaGlnaF9odV90aHJlc2hvbGQiKSwKICAgICAgICAgICAgImxvd19odV90aHJlc2hvbGQiOiBwLmdldCgibG93X2h1X3RocmVzaG9sZCIpLAogICAgICAgIH0KCgpkZWYgX2NvbXBvbmVudF9zdGF0cyhtYXNrLCBwbGFxdWVfbGFiZWw9Mik6CiAgICBwbGFxdWUgPSBucC5hc2FycmF5KG1hc2spID09IHBsYXF1ZV9sYWJlbAogICAgaWYgbm90IG5wLmFueShwbGFxdWUpOgogICAgICAgIHJldHVybiAwLCAwLjAKICAgIGxhYmVscywgbiA9IG5kaS5sYWJlbChwbGFxdWUpCiAgICBzaXplcyA9IG5wLmJpbmNvdW50KGxhYmVscy5yYXZlbCgpKQogICAgaWYgbGVuKHNpemVzKSA8PSAxOgogICAgICAgIHJldHVybiBpbnQobiksIDAuMAogICAgbGFyZ2VzdCA9IGZsb2F0KG5wLm1heChzaXplc1sxOl0pKQogICAgdG90YWwgPSBmbG9hdChucC5zdW0oc2l6ZXNbMTpdKSkKICAgIHJldHVybiBpbnQobiksIGxhcmdlc3QgLyB0b3RhbCBpZiB0b3RhbCA+IDAgZWxzZSAwLjAKCgpkZWYgX2h1X3N0YXRzKHZvbHVtZSwgbWFzaywgcGxhcXVlX2xhYmVsPTIpOgogICAgdmFscyA9IG5wLmFzYXJyYXkodm9sdW1lKVtucC5hc2FycmF5KG1hc2spID09IHBsYXF1ZV9sYWJlbF0KICAgIGlmIHZhbHMuc2l6ZSA9PSAwOgogICAgICAgIHJldHVybiBOb25lLCBOb25lCiAgICByZXR1cm4gZmxvYXQobnAubWVhbih2YWxzKSksIGZsb2F0KG5wLm1lZGlhbih2YWxzKSkKCgpkZWYgX3Njb3JlX2NhbmRpZGF0ZShyZXRhaW5lZF9mcmFjdGlvbjogZmxvYXQsCiAgICAgICAgICAgICAgICAgICAgIHJlbW92ZWRfZnJhY3Rpb246IGZsb2F0LAogICAgICAgICAgICAgICAgICAgICBjb21wb25lbnRzX2FmdGVyOiBpbnQsCiAgICAgICAgICAgICAgICAgICAgIGNvbXBvbmVudHNfYmVmb3JlOiBpbnQsCiAgICAgICAgICAgICAgICAgICAgIGxhcmdlc3RfY29tcG9uZW50X2ZyYWN0aW9uOiBmbG9hdCwKICAgICAgICAgICAgICAgICAgICAgbWVhbl9odTogT3B0aW9uYWxbZmxvYXRdLAogICAgICAgICAgICAgICAgICAgICBlcm9kZV9jb3JlOiBib29sKSAtPiB0dXBsZVtmbG9hdCwgc3RyXToKICAgICIiIkhldXJpc3RpYyBzY29yZS4gSGlnaGVyIGlzIGJldHRlci4KCiAgICBSZWplY3RzIGV4dHJlbWUgc2V0dGluZ3MgdGhhdCByZXRhaW4gYWxtb3N0IGV2ZXJ5dGhpbmcgb3IgZGVsZXRlIHRvbyBtdWNoLgogICAgUHJlZmVycmVkIGJlaGF2aW9yIGlzIG1vZGVzdCBib3VuZGFyeSBjbGVhbnVwLCBmZXdlciB0aW55IGNvbXBvbmVudHMsIGFuZAogICAgc3RhYmxlIHBsYXF1ZSBjb3JlIHByZXNlcnZhdGlvbi4KICAgICIiIgogICAgcmVhc29ucyA9IFtdCiAgICBpZiByZXRhaW5lZF9mcmFjdGlvbiA8PSAwOgogICAgICAgIHJlYXNvbnMuYXBwZW5kKCJkZWxldGVkX2FsbF9wbGFxdWUiKQogICAgaWYgcmV0YWluZWRfZnJhY3Rpb24gPCAwLjM1OgogICAgICAgIHJlYXNvbnMuYXBwZW5kKCJvdmVyX2FnZ3Jlc3NpdmVfcmV0YWluc191bmRlcl8zNXBjdCIpCiAgICBpZiByZXRhaW5lZF9mcmFjdGlvbiA+IDAuOTk1OgogICAgICAgIHJlYXNvbnMuYXBwZW5kKCJub19lZmZlY3QiKQoKICAgICMgVGFyZ2V0IGEgbW9kZXJhdGUgY2xlYW51cC4gVGhpcyBpcyBpbnRlbnRpb25hbGx5IGJyb2FkIGJlY2F1c2UgZGF0YSB2YXJ5LgogICAgdGFyZ2V0X3JlbW92ZWQgPSAwLjE1CiAgICBzY29yZSA9IDEwMC4wCiAgICBzY29yZSAtPSAxODAuMCAqIGFicyhyZW1vdmVkX2ZyYWN0aW9uIC0gdGFyZ2V0X3JlbW92ZWQpCgogICAgIyBSZXdhcmQgcmVtb3ZpbmcgZnJhZ21lbnRlZCBub2lzZSBidXQgZG8gbm90IGRlbWFuZCBvbmUgY29tcG9uZW50LgogICAgaWYgY29tcG9uZW50c19iZWZvcmUgPiAwOgogICAgICAgIGNvbXBvbmVudF9yZWR1Y3Rpb24gPSBtYXgoMCwgY29tcG9uZW50c19iZWZvcmUgLSBjb21wb25lbnRzX2FmdGVyKSAvIGNvbXBvbmVudHNfYmVmb3JlCiAgICAgICAgc2NvcmUgKz0gMTguMCAqIGNvbXBvbmVudF9yZWR1Y3Rpb24KCiAgICAjIFJld2FyZCBtYXNrcyB3aXRoIGEgZG9taW5hbnQgcGxhcXVlIGNvbXBvbmVudCBhZnRlciByZWZpbmVtZW50LgogICAgc2NvcmUgKz0gMTguMCAqIGxhcmdlc3RfY29tcG9uZW50X2ZyYWN0aW9uCgogICAgIyBQZW5hbGl6ZSBjb3JlIGVyb3Npb24gZm9yIHRoZSBtYWluIHJlZmluZWQgZXN0aW1hdGU7IGNvcmUgaXMgcmVwb3J0ZWQgc2VwYXJhdGVseS4KICAgIGlmIGVyb2RlX2NvcmU6CiAgICAgICAgc2NvcmUgLT0gMTUuMAoKICAgICMgU29mdCBIVSBwbGF1c2liaWxpdHk6IGF2b2lkIGV4dHJlbWVseSBsb3cvaGlnaCBtZWFuIHBsYXF1ZSBIVSB3aGVuIHRocmVzaG9sZGluZyBpcyB1c2VkLgogICAgaWYgbWVhbl9odSBpcyBub3QgTm9uZToKICAgICAgICBpZiBtZWFuX2h1IDwgLTE1MCBvciBtZWFuX2h1ID4gMTEwMDoKICAgICAgICAgICAgc2NvcmUgLT0gMjAuMAoKICAgIGlmIHJlYXNvbnM6CiAgICAgICAgc2NvcmUgLT0gMTAwMC4wCiAgICByZXR1cm4gZmxvYXQoc2NvcmUpLCAiOyIuam9pbihyZWFzb25zKQoKCmRlZiB0dW5lX2JvdW5kYXJ5X2Zvcl9yZXBvcnQocmVwb3J0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhcmFtZXRlcl9ncmlkOiBPcHRpb25hbFtNYXBwaW5nW3N0ciwgU2VxdWVuY2VdXSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWluX3JldGFpbmVkX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMzUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3JldGFpbmVkX2ZyYWN0aW9uOiBmbG9hdCA9IDAuOTk1KSAtPiBsaXN0W1R1bmluZ1Jvd106CiAgICByb3dzOiBsaXN0W1R1bmluZ1Jvd10gPSBbXQogICAgcmF3X3RwdiA9IGZsb2F0KHJlcG9ydC50cHZfbW0zKQogICAgY29tcG9uZW50c19iZWZvcmUsIF8gPSBfY29tcG9uZW50X3N0YXRzKHJlcG9ydC5tYXNrKQoKICAgIGZvciBjaWQsIHBhcmFtcyBpbiBfcGFyYW1ldGVyX2NhbmRpZGF0ZXMocGFyYW1ldGVyX2dyaWQpOgogICAgICAgIHJlZiA9IHJlZmluZV9wbGFxdWVfbWFzaygKICAgICAgICAgICAgdm9sdW1lPXJlcG9ydC52b2x1bWUsCiAgICAgICAgICAgIG1hc2s9cmVwb3J0Lm1hc2ssCiAgICAgICAgICAgIHNwYWNpbmc9cmVwb3J0Lm1hc2tfaW1hZ2UuR2V0U3BhY2luZygpLAogICAgICAgICAgICAqKnBhcmFtcywKICAgICAgICApCiAgICAgICAgcmVmaW5lZF90cHYgPSBmbG9hdChyZWYucmVmaW5lZF90cHZfbW0zKQogICAgICAgIHJldGFpbmVkID0gcmVmaW5lZF90cHYgLyByYXdfdHB2IGlmIHJhd190cHYgPiAwIGVsc2UgMC4wCiAgICAgICAgcmVtb3ZlZF9mcmFjdGlvbiA9IDEuMCAtIHJldGFpbmVkCiAgICAgICAgY29tcG9uZW50c19hZnRlciwgbGFyZ2VzdF9mcmFjID0gX2NvbXBvbmVudF9zdGF0cyhyZWYucmVmaW5lZF9tYXNrKQogICAgICAgIG1lYW5faHUsIG1lZGlhbl9odSA9IF9odV9zdGF0cyhyZXBvcnQudm9sdW1lLCByZWYucmVmaW5lZF9tYXNrKQogICAgICAgIHNjb3JlLCByZWplY3QgPSBfc2NvcmVfY2FuZGlkYXRlKAogICAgICAgICAgICByZXRhaW5lZCwgcmVtb3ZlZF9mcmFjdGlvbiwgY29tcG9uZW50c19hZnRlciwgY29tcG9uZW50c19iZWZvcmUsCiAgICAgICAgICAgIGxhcmdlc3RfZnJhYywgbWVhbl9odSwgcGFyYW1zWyJlcm9kZV9jb3JlIl0KICAgICAgICApCiAgICAgICAgaWYgcmV0YWluZWQgPCBtaW5fcmV0YWluZWRfZnJhY3Rpb24gYW5kICJvdmVyX2FnZ3Jlc3NpdmUiIG5vdCBpbiByZWplY3Q6CiAgICAgICAgICAgIHJlamVjdCA9IChyZWplY3QgKyAiOyIgaWYgcmVqZWN0IGVsc2UgIiIpICsgImJlbG93X21pbl9yZXRhaW5lZF9mcmFjdGlvbiIKICAgICAgICAgICAgc2NvcmUgLT0gMTAwMAogICAgICAgIGlmIHJldGFpbmVkID4gbWF4X3JldGFpbmVkX2ZyYWN0aW9uIGFuZCAibm9fZWZmZWN0IiBub3QgaW4gcmVqZWN0OgogICAgICAgICAgICByZWplY3QgPSAocmVqZWN0ICsgIjsiIGlmIHJlamVjdCBlbHNlICIiKSArICJhYm92ZV9tYXhfcmV0YWluZWRfZnJhY3Rpb24iCiAgICAgICAgICAgIHNjb3JlIC09IDEwMDAKICAgICAgICByb3dzLmFwcGVuZChUdW5pbmdSb3coCiAgICAgICAgICAgIHZlc3NlbD1yZXBvcnQubmFtZSwKICAgICAgICAgICAgY2FuZGlkYXRlX2lkPWNpZCwKICAgICAgICAgICAgbWluX2NvbXBvbmVudF92b3hlbHM9cGFyYW1zWyJtaW5fY29tcG9uZW50X3ZveGVscyJdLAogICAgICAgICAgICBsdW1lbl9kaXN0YW5jZV92b3hlbHM9cGFyYW1zWyJsdW1lbl9kaXN0YW5jZV92b3hlbHMiXSBpZiBwYXJhbXNbInRyaW1fbHVtZW5fYWRqYWNlbnQiXSBlbHNlIDAsCiAgICAgICAgICAgIGhpZ2hfaHVfdGhyZXNob2xkPXBhcmFtc1siaGlnaF9odV90aHJlc2hvbGQiXSwKICAgICAgICAgICAgbG93X2h1X3RocmVzaG9sZD1wYXJhbXNbImxvd19odV90aHJlc2hvbGQiXSwKICAgICAgICAgICAgZXJvZGVfY29yZT1wYXJhbXNbImVyb2RlX2NvcmUiXSwKICAgICAgICAgICAgZXJvc2lvbl9pdGVyYXRpb25zPXBhcmFtc1siZXJvc2lvbl9pdGVyYXRpb25zIl0sCiAgICAgICAgICAgIHJhd190cHZfbW0zPXJhd190cHYsCiAgICAgICAgICAgIHJlZmluZWRfdHB2X21tMz1yZWZpbmVkX3RwdiwKICAgICAgICAgICAgcmVtb3ZlZF9tbTM9ZmxvYXQocmVmLnJlbW92ZWRfdm9sdW1lX21tMyksCiAgICAgICAgICAgIHJldGFpbmVkX2ZyYWN0aW9uPWZsb2F0KHJldGFpbmVkKSwKICAgICAgICAgICAgcmVtb3ZlZF9mcmFjdGlvbj1mbG9hdChyZW1vdmVkX2ZyYWN0aW9uKSwKICAgICAgICAgICAgY29tcG9uZW50c19iZWZvcmU9Y29tcG9uZW50c19iZWZvcmUsCiAgICAgICAgICAgIGNvbXBvbmVudHNfYWZ0ZXI9Y29tcG9uZW50c19hZnRlciwKICAgICAgICAgICAgbGFyZ2VzdF9jb21wb25lbnRfZnJhY3Rpb249ZmxvYXQobGFyZ2VzdF9mcmFjKSwKICAgICAgICAgICAgbWVhbl9odT1tZWFuX2h1LAogICAgICAgICAgICBtZWRpYW5faHU9bWVkaWFuX2h1LAogICAgICAgICAgICBzY29yZT1zY29yZSwKICAgICAgICAgICAgcmVqZWN0X3JlYXNvbj1yZWplY3QsCiAgICAgICAgKSkKICAgIHJldHVybiByb3dzCgoKZGVmIHR1bmVfYm91bmRhcnlfcGFyYW1ldGVycyhyZXBvcnRzOiBJdGVyYWJsZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICBwYXJhbWV0ZXJfZ3JpZDogT3B0aW9uYWxbTWFwcGluZ1tzdHIsIFNlcXVlbmNlXV0gPSBOb25lKSAtPiBUdW5pbmdSZXN1bHQ6CiAgICBhbGxfcm93czogbGlzdFtUdW5pbmdSb3ddID0gW10KICAgIGJlc3RfYnlfdmVzc2VsOiBEaWN0W3N0ciwgVHVuaW5nUm93XSA9IHt9CiAgICBmb3IgcmVwb3J0IGluIHJlcG9ydHM6CiAgICAgICAgcm93cyA9IHR1bmVfYm91bmRhcnlfZm9yX3JlcG9ydChyZXBvcnQsIHBhcmFtZXRlcl9ncmlkPXBhcmFtZXRlcl9ncmlkKQogICAgICAgIGFsbF9yb3dzLmV4dGVuZChyb3dzKQogICAgICAgIHZhbGlkID0gW3IgZm9yIHIgaW4gcm93cyBpZiBub3Qgci5yZWplY3RfcmVhc29uXQogICAgICAgIGJlc3QgPSBtYXgodmFsaWQgb3Igcm93cywga2V5PWxhbWJkYSByOiByLnNjb3JlKQogICAgICAgIGJlc3RfYnlfdmVzc2VsW3JlcG9ydC5uYW1lXSA9IGJlc3QKCiAgICAjIENob29zZSBhIHNpbmdsZSBnbG9iYWwgcGFyYW1ldGVyIHNldCBieSBzdW1taW5nIHNjb3JlcyBhY3Jvc3MgdmVzc2Vscy4KICAgIGdyb3VwZWQgPSB7fQogICAgZm9yIHIgaW4gYWxsX3Jvd3M6CiAgICAgICAga2V5ID0gKAogICAgICAgICAgICByLm1pbl9jb21wb25lbnRfdm94ZWxzLCByLmx1bWVuX2Rpc3RhbmNlX3ZveGVscywKICAgICAgICAgICAgci5oaWdoX2h1X3RocmVzaG9sZCwgci5sb3dfaHVfdGhyZXNob2xkLAogICAgICAgICAgICByLmVyb2RlX2NvcmUsIHIuZXJvc2lvbl9pdGVyYXRpb25zLAogICAgICAgICkKICAgICAgICBncm91cGVkLnNldGRlZmF1bHQoa2V5LCAwLjApCiAgICAgICAgZ3JvdXBlZFtrZXldICs9IHIuc2NvcmUKICAgIGJlc3Rfa2V5ID0gbWF4KGdyb3VwZWQsIGtleT1ncm91cGVkLmdldCkgaWYgZ3JvdXBlZCBlbHNlICgxMCwgMSwgTm9uZSwgTm9uZSwgRmFsc2UsIDEpCiAgICBzZWxlY3RlZF9wYXJhbXMgPSB7CiAgICAgICAgInJlbW92ZV9zbWFsbCI6IFRydWUsCiAgICAgICAgIm1pbl9jb21wb25lbnRfdm94ZWxzIjogYmVzdF9rZXlbMF0sCiAgICAgICAgInRyaW1fbHVtZW5fYWRqYWNlbnQiOiBiZXN0X2tleVsxXSA+IDAsCiAgICAgICAgImx1bWVuX2Rpc3RhbmNlX3ZveGVscyI6IGJlc3Rfa2V5WzFdLAogICAgICAgICJlcm9kZV9jb3JlIjogYmVzdF9rZXlbNF0sCiAgICAgICAgImVyb3Npb25faXRlcmF0aW9ucyI6IGJlc3Rfa2V5WzVdLAogICAgICAgICJoaWdoX2h1X3RocmVzaG9sZCI6IGJlc3Rfa2V5WzJdLAogICAgICAgICJsb3dfaHVfdGhyZXNob2xkIjogYmVzdF9rZXlbM10sCiAgICB9CiAgICBub3RlcyA9ICgKICAgICAgICAiVW5zdXBlcnZpc2VkIHR1bmluZzogc2VsZWN0ZWQgYnkgaGV1cmlzdGljIHNjb3JlIHVzaW5nIHJldGFpbmVkIFRQViwgcmVtb3ZlZCBmcmFjdGlvbiwgIgogICAgICAgICJmcmFnbWVudGF0aW9uIHJlZHVjdGlvbiwgbGFyZ2VzdC1jb21wb25lbnQgZnJhY3Rpb24sIGFuZCBIVSBzYW5pdHkgY2hlY2tzLiAiCiAgICAgICAgIlVzZSBleHBlcnQgdmlzdWFsIFFDIGJlZm9yZSBpbnRlcnByZXRpbmcgcmVzdWx0cy4iCiAgICApCiAgICByZXR1cm4gVHVuaW5nUmVzdWx0KHJvd3M9YWxsX3Jvd3MsIGJlc3RfYnlfdmVzc2VsPWJlc3RfYnlfdmVzc2VsLCBzZWxlY3RlZF9wYXJhbXM9c2VsZWN0ZWRfcGFyYW1zLCBub3Rlcz1ub3RlcykKCgpkZWYgYXBwbHlfc2VsZWN0ZWRfcmVmaW5lbWVudChyZXBvcnRzOiBJdGVyYWJsZSwgc2VsZWN0ZWRfcGFyYW1zOiBNYXBwaW5nKSAtPiBEaWN0W3N0ciwgb2JqZWN0XToKICAgIG91dCA9IHt9CiAgICBmb3IgcmVwb3J0IGluIHJlcG9ydHM6CiAgICAgICAgb3V0W3JlcG9ydC5uYW1lXSA9IHJlZmluZV9wbGFxdWVfbWFzaygKICAgICAgICAgICAgdm9sdW1lPXJlcG9ydC52b2x1bWUsCiAgICAgICAgICAgIG1hc2s9cmVwb3J0Lm1hc2ssCiAgICAgICAgICAgIHNwYWNpbmc9cmVwb3J0Lm1hc2tfaW1hZ2UuR2V0U3BhY2luZygpLAogICAgICAgICAgICAqKmRpY3Qoc2VsZWN0ZWRfcGFyYW1zKSwKICAgICAgICApCiAgICByZXR1cm4gb3V0CgoKZGVmIHdyaXRlX3R1bmluZ19odG1sX3JlcG9ydCh0dW5pbmdfcmVzdWx0OiBUdW5pbmdSZXN1bHQsIG91dHB1dF9odG1sLCB0aXRsZT0iT3BlblBsYXF1ZSBCb3VuZGFyeSBQYXJhbWV0ZXIgVHVuaW5nIFJlcG9ydCIpIC0+IFBhdGg6CiAgICBvdXRwdXRfaHRtbCA9IFBhdGgob3V0cHV0X2h0bWwpCiAgICBvdXRwdXRfaHRtbC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIGRlZiBmbXQoeCk6CiAgICAgICAgaWYgeCBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gIiIKICAgICAgICBpZiBpc2luc3RhbmNlKHgsIGJvb2wpOgogICAgICAgICAgICByZXR1cm4gInllcyIgaWYgeCBlbHNlICJubyIKICAgICAgICBpZiBpc2luc3RhbmNlKHgsIGZsb2F0KToKICAgICAgICAgICAgcmV0dXJuIGYie3g6LjNmfSIKICAgICAgICByZXR1cm4gc3RyKHgpCgogICAgcm93cyA9IHNvcnRlZCh0dW5pbmdfcmVzdWx0LnJvd3MsIGtleT1sYW1iZGEgcjogKHIudmVzc2VsLCAtci5zY29yZSkpCiAgICBib2R5X3Jvd3MgPSBbXQogICAgZm9yIHIgaW4gcm93czoKICAgICAgICBjbHMgPSAiIGNsYXNzPSdyZWplY3RlZCciIGlmIHIucmVqZWN0X3JlYXNvbiBlbHNlICIiCiAgICAgICAgYm9keV9yb3dzLmFwcGVuZCgKICAgICAgICAgICAgZiI8dHJ7Y2xzfT48dGQ+e3IudmVzc2VsfTwvdGQ+PHRkPntyLmNhbmRpZGF0ZV9pZH08L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e3Iuc2NvcmU6LjJmfTwvdGQ+PHRkPntyLm1pbl9jb21wb25lbnRfdm94ZWxzfTwvdGQ+IgogICAgICAgICAgICBmIjx0ZD57ci5sdW1lbl9kaXN0YW5jZV92b3hlbHN9PC90ZD48dGQ+e2ZtdChyLmhpZ2hfaHVfdGhyZXNob2xkKX08L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e2ZtdChyLmxvd19odV90aHJlc2hvbGQpfTwvdGQ+PHRkPntmbXQoci5lcm9kZV9jb3JlKX08L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e3IucmF3X3Rwdl9tbTM6LjJmfTwvdGQ+PHRkPntyLnJlZmluZWRfdHB2X21tMzouMmZ9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPnsxMDAqci5yZXRhaW5lZF9mcmFjdGlvbjouMWZ9JTwvdGQ+PHRkPntyLmNvbXBvbmVudHNfYmVmb3JlfTwvdGQ+IgogICAgICAgICAgICBmIjx0ZD57ci5jb21wb25lbnRzX2FmdGVyfTwvdGQ+PHRkPntyLmxhcmdlc3RfY29tcG9uZW50X2ZyYWN0aW9uOi4zZn08L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e2ZtdChyLm1lYW5faHUpfTwvdGQ+PHRkPntyLnJlamVjdF9yZWFzb259PC90ZD48L3RyPiIKICAgICAgICApCgogICAgYmVzdF9yb3dzID0gW10KICAgIGZvciB2ZXNzZWwsIHIgaW4gdHVuaW5nX3Jlc3VsdC5iZXN0X2J5X3Zlc3NlbC5pdGVtcygpOgogICAgICAgIGJlc3Rfcm93cy5hcHBlbmQoCiAgICAgICAgICAgIGYiPHRyPjx0ZD57dmVzc2VsfTwvdGQ+PHRkPntyLmNhbmRpZGF0ZV9pZH08L3RkPjx0ZD57ci5zY29yZTouMmZ9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntyLm1pbl9jb21wb25lbnRfdm94ZWxzfTwvdGQ+PHRkPntyLmx1bWVuX2Rpc3RhbmNlX3ZveGVsc308L3RkPiIKICAgICAgICAgICAgZiI8dGQ+e2ZtdChyLmhpZ2hfaHVfdGhyZXNob2xkKX08L3RkPjx0ZD57Zm10KHIubG93X2h1X3RocmVzaG9sZCl9PC90ZD4iCiAgICAgICAgICAgIGYiPHRkPntmbXQoci5lcm9kZV9jb3JlKX08L3RkPjx0ZD57ci5yZWZpbmVkX3Rwdl9tbTM6LjJmfTwvdGQ+IgogICAgICAgICAgICBmIjx0ZD57MTAwKnIucmV0YWluZWRfZnJhY3Rpb246LjFmfSU8L3RkPjwvdHI+IgogICAgICAgICkKCiAgICBwYXJhbXNfanNvbiA9IGpzb24uZHVtcHModHVuaW5nX3Jlc3VsdC5zZWxlY3RlZF9wYXJhbXMsIGluZGVudD0yKQogICAgaHRtbCA9IGYiIiI8IWRvY3R5cGUgaHRtbD4KPGh0bWwgbGFuZz0iZW4iPjxoZWFkPjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij48dGl0bGU+e3RpdGxlfTwvdGl0bGU+CjxzdHlsZT4KYm9keSB7eyBmb250LWZhbWlseTogLWFwcGxlLXN5c3RlbSwgQmxpbmtNYWNTeXN0ZW1Gb250LCBTZWdvZSBVSSwgc2Fucy1zZXJpZjsgbWFyZ2luOiAzMnB4OyBjb2xvcjogIzIyMjsgfX0KdGFibGUge3sgYm9yZGVyLWNvbGxhcHNlOiBjb2xsYXBzZTsgd2lkdGg6IDEwMCU7IG1hcmdpbjogMTZweCAwIDI4cHg7IGZvbnQtc2l6ZTogMC45MnJlbTsgfX0KdGgsIHRkIHt7IGJvcmRlcjogMXB4IHNvbGlkICNkZGQ7IHBhZGRpbmc6IDZweCA4cHg7IHRleHQtYWxpZ246IHJpZ2h0OyB9fQp0aDpmaXJzdC1jaGlsZCwgdGQ6Zmlyc3QtY2hpbGQge3sgdGV4dC1hbGlnbjogbGVmdDsgfX0gdGgge3sgYmFja2dyb3VuZDogI2Y2ZjZmNjsgcG9zaXRpb246IHN0aWNreTsgdG9wOiAwOyB9fQoucmVqZWN0ZWQge3sgY29sb3I6ICM4ODg7IGJhY2tncm91bmQ6ICNmYWZhZmE7IH19Ci5ub3RpY2Uge3sgcGFkZGluZzogMTJweCAxNHB4OyBiYWNrZ3JvdW5kOiAjZmZmM2NkOyBib3JkZXI6IDFweCBzb2xpZCAjZmZlMDhhOyBib3JkZXItcmFkaXVzOiA4cHg7IH19CnByZSB7eyBiYWNrZ3JvdW5kOiAjZjdmN2Y3OyBwYWRkaW5nOiAxMnB4OyBib3JkZXItcmFkaXVzOiA4cHg7IG92ZXJmbG93LXg6IGF1dG87IH19Cjwvc3R5bGU+PC9oZWFkPjxib2R5Pgo8aDE+e3RpdGxlfTwvaDE+CjxwIGNsYXNzPSJub3RpY2UiPjxzdHJvbmc+UmVzZWFyY2ggdXNlIG9ubHkuPC9zdHJvbmc+IFRoaXMgaXMgdW5zdXBlcnZpc2VkIHBhcmFtZXRlciB0dW5pbmcsIG5vdCBjbGluaWNhbCB2YWxpZGF0aW9uLiBVc2UgdmlzdWFsIFFDLjwvcD4KPGgyPlNlbGVjdGVkIGdsb2JhbCBwYXJhbWV0ZXJzPC9oMj48cHJlPntwYXJhbXNfanNvbn08L3ByZT4KPHA+e3R1bmluZ19yZXN1bHQubm90ZXN9PC9wPgo8aDI+QmVzdCBjYW5kaWRhdGUgYnkgdmVzc2VsPC9oMj4KPHRhYmxlPjx0aGVhZD48dHI+PHRoPlZlc3NlbDwvdGg+PHRoPkNhbmRpZGF0ZTwvdGg+PHRoPlNjb3JlPC90aD48dGg+TWluIGNvbXAgdm94PC90aD48dGg+THVtZW4gZGlzdDwvdGg+PHRoPkhpZ2ggSFU8L3RoPjx0aD5Mb3cgSFU8L3RoPjx0aD5Fcm9kZTwvdGg+PHRoPlJlZmluZWQgVFBWPC90aD48dGg+UmV0YWluZWQ8L3RoPjwvdHI+PC90aGVhZD48dGJvZHk+eycnLmpvaW4oYmVzdF9yb3dzKX08L3Rib2R5PjwvdGFibGU+CjxoMj5BbGwgY2FuZGlkYXRlczwvaDI+Cjx0YWJsZT48dGhlYWQ+PHRyPjx0aD5WZXNzZWw8L3RoPjx0aD5JRDwvdGg+PHRoPlNjb3JlPC90aD48dGg+TWluIGNvbXAgdm94PC90aD48dGg+THVtZW4gZGlzdDwvdGg+PHRoPkhpZ2ggSFU8L3RoPjx0aD5Mb3cgSFU8L3RoPjx0aD5Fcm9kZTwvdGg+PHRoPlJhdyBUUFY8L3RoPjx0aD5SZWZpbmVkIFRQVjwvdGg+PHRoPlJldGFpbmVkPC90aD48dGg+Q29tcCBiZWZvcmU8L3RoPjx0aD5Db21wIGFmdGVyPC90aD48dGg+TGFyZ2VzdCBjb21wIGZyYWM8L3RoPjx0aD5NZWFuIEhVPC90aD48dGg+UmVqZWN0IHJlYXNvbjwvdGg+PC90cj48L3RoZWFkPjx0Ym9keT57Jycuam9pbihib2R5X3Jvd3MpfTwvdGJvZHk+PC90YWJsZT4KPC9ib2R5PjwvaHRtbD4iIiIKICAgIG91dHB1dF9odG1sLndyaXRlX3RleHQoaHRtbCwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHJldHVybiBvdXRwdXRfaHRtbAo=',
    '__init__.py': 'dHJ5OgogICAgZnJvbSAuYXJ0ZXJ5X2RldGVjdGlvbiBpbXBvcnQgZGV0ZWN0X2FydGVyeV9zZXJpZXMsIGxvYWRfZGV0ZWN0ZWRfYXJ0ZXJpZXMKZXhjZXB0IEV4Y2VwdGlvbjoKICAgIHBhc3MKdHJ5OgogICAgZnJvbSAudW5jZXJ0YWludHkgaW1wb3J0IG1ha2VfdHB2X3VuY2VydGFpbnR5X3N1bW1hcnkKZXhjZXB0IEV4Y2VwdGlvbjoKICAgIHBhc3MKdHJ5OgogICAgZnJvbSAucmVwb3J0IGltcG9ydCB3cml0ZV9odG1sX3JlcG9ydApleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwoKdHJ5OgogICAgZnJvbSAudHVuaW5nIGltcG9ydCAoCiAgICAgICAgdHVuZV9ib3VuZGFyeV9wYXJhbWV0ZXJzLAogICAgICAgIGFwcGx5X3NlbGVjdGVkX3JlZmluZW1lbnQsCiAgICAgICAgd3JpdGVfdHVuaW5nX2h0bWxfcmVwb3J0LAogICAgKQpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwo=',
}

for filename, payload in MODULES_B64.items():
    path = PKG_DIR / filename
    path.write_bytes(base64.b64decode(payload))
    print('Wrote', path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import openplaque
print('OpenPlaque package:', openplaque.__file__)


In [ ]:
# 4. Configure nnU-Net environment and verify GPU
import os
from pathlib import Path

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_results'

for key in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results']:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)
    print(key, '=', os.environ[key])

!nvidia-smi


In [ ]:
# 5. Extract nnU-Net model weights
import zipfile
from pathlib import Path

model_target = Path('/content/nnUNet_results/Dataset001_CCTA_DHM')

if model_target.exists():
    print('Model already extracted:', model_target)
else:
    if not MODEL_ZIP.exists():
        raise FileNotFoundError(f'Missing model ZIP: {MODEL_ZIP}')
    print('Extracting model ZIP...')
    with zipfile.ZipFile(MODEL_ZIP) as z:
        z.extractall('/content/nnUNet_results')
    print('Extracted model to /content/nnUNet_results')

!find /content/nnUNet_results -maxdepth 4


In [ ]:
# 6. Load full DICOM study
from openplaque.study import OpenPlaqueStudy

if not STUDY_ZIP.exists():
    raise FileNotFoundError(f'Missing study ZIP: {STUDY_ZIP}')

study = OpenPlaqueStudy(str(STUDY_ZIP))
study.summary()


In [ ]:
# 7. Automatically detect LAD/RCA/LCX curved reformat series
from openplaque.artery_detection import detect_artery_series

arteries = detect_artery_series(study)
print('Detected artery series:')
for vessel, series_number in arteries.items():
    print(f'  {vessel}: {series_number}')

required = {'LAD', 'RCA', 'LCX'}
missing = required - set(arteries)
if missing:
    raise RuntimeError(f'Missing detected artery series: {missing}. You can manually set arteries = {{"RCA": 1035, "LCX": 1039, "LAD": 1043}} and rerun from here.')


In [ ]:
# 8. Load detected curved coronary reformats
image_lad, volume_lad, _ = study.load_series(arteries['LAD'])
image_rca, volume_rca, _ = study.load_series(arteries['RCA'])
image_lcx, volume_lcx, _ = study.load_series(arteries['LCX'])

print('LAD:', volume_lad.shape, image_lad.GetSpacing())
print('RCA:', volume_rca.shape, image_rca.GetSpacing())
print('LCX:', volume_lcx.shape, image_lcx.GetSpacing())


In [ ]:
# 9. Quick visual QC of loaded reformats
import matplotlib.pyplot as plt

def show_mid(volume, title):
    z = volume.shape[0] // 2
    plt.figure(figsize=(6, 6))
    plt.imshow(volume[z], cmap='gray', vmin=-200, vmax=800)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_mid(volume_lad, 'LAD curved reformat')
show_mid(volume_rca, 'RCA curved reformat')
show_mid(volume_lcx, 'LCX curved reformat')


In [ ]:
# 10. Run plaque segmentation on each artery
# Requires GPU runtime and extracted nnU-Net model.
from openplaque.segmentation import segment_vessel

lad_report = segment_vessel(image_lad, volume_lad, 'LAD')
lad_report.summary()
lad_report.show_overlay(label=2)

rca_report = segment_vessel(image_rca, volume_rca, 'RCA')
rca_report.summary()
rca_report.show_overlay(label=2)

lcx_report = segment_vessel(image_lcx, volume_lcx, 'LCX')
lcx_report.summary()
lcx_report.show_overlay(label=2)

reports = [lad_report, rca_report, lcx_report]


In [ ]:
# 11. Raw TPV summary
raw_total_tpv = sum(r.tpv_mm3 for r in reports)
print('Raw plaque volume by vessel')
print('-' * 40)
for r in reports:
    print(f'{r.name}: {r.tpv_mm3:.2f} mm³ ({r.plaque_voxels} voxels)')
print('-' * 40)
print(f'RAW TOTAL PLAQUE VOLUME: {raw_total_tpv:.2f} mm³')


## Boundary-refinement parameter tuning

The tuning is unsupervised. It compares candidate settings for small-component removal, lumen-adjacent trimming, optional HU thresholds, and core erosion. The score favors cleanup that is conservative but not destructive.


In [ ]:
# 12. Run boundary-refinement parameter tuning
from openplaque.tuning import (
    tune_boundary_parameters,
    apply_selected_refinement,
    write_tuning_html_report,
)

tuning = tune_boundary_parameters(reports)

print('Selected global parameters:')
for k, v in tuning.selected_params.items():
    print(f'  {k}: {v}')

print('\nBest by vessel:')
for vessel, row in tuning.best_by_vessel.items():
    print(f'{vessel}: candidate {row.candidate_id}, score={row.score:.2f}, refined={row.refined_tpv_mm3:.2f} mm³, retained={100*row.retained_fraction:.1f}%')


In [ ]:
# 13. Save tuning tables and tuning HTML report
csv_path = tuning.save_csv(SAVE_DIR / 'boundary_tuning_results.csv')
json_path = tuning.save_json(SAVE_DIR / 'boundary_tuning_results.json')
tuning_html_path = write_tuning_html_report(tuning, SAVE_DIR / 'boundary_tuning_report.html')

print('Saved:', csv_path)
print('Saved:', json_path)
print('Saved:', tuning_html_path)


In [ ]:
# 14. Inspect top candidates in a table
import pandas as pd

df = pd.DataFrame(tuning.to_rows())
cols = [
    'vessel', 'candidate_id', 'score', 'min_component_voxels',
    'lumen_distance_voxels', 'high_hu_threshold', 'low_hu_threshold',
    'erode_core', 'raw_tpv_mm3', 'refined_tpv_mm3',
    'retained_fraction', 'components_before', 'components_after',
    'largest_component_fraction', 'mean_hu', 'reject_reason'
]

for vessel in df['vessel'].unique():
    print('\n' + '='*80)
    print(vessel)
    display(df[df['vessel'] == vessel].sort_values('score', ascending=False)[cols].head(12))


In [ ]:
# 15. Plot tuning sensitivity
import matplotlib.pyplot as plt

for vessel in df['vessel'].unique():
    sub = df[df['vessel'] == vessel].copy().sort_values('score', ascending=False).head(40)
    plt.figure(figsize=(9, 4))
    plt.plot(range(len(sub)), sub['refined_tpv_mm3'].values, marker='o')
    plt.title(f'{vessel}: refined TPV among top 40 candidates')
    plt.xlabel('Candidate rank by score')
    plt.ylabel('Refined TPV mm³')
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
# 16. Apply selected tuned refinement and generate final TPV HTML report
from openplaque.boundary import refine_plaque_mask
from openplaque.uncertainty import make_tpv_uncertainty_summary
from openplaque.report import write_html_report

tuned_refined = apply_selected_refinement(reports, tuning.selected_params)

# High-confidence core is intentionally conservative for the lower bound.
core_results = {}
for report in reports:
    core_results[report.name] = refine_plaque_mask(
        volume=report.volume,
        mask=report.mask,
        spacing=report.mask_image.GetSpacing(),
        remove_small=True,
        min_component_voxels=max(10, tuning.selected_params.get('min_component_voxels', 10)),
        trim_lumen_adjacent=True,
        lumen_distance_voxels=max(1, tuning.selected_params.get('lumen_distance_voxels', 1)),
        erode_core=True,
        erosion_iterations=1,
        high_hu_threshold=tuning.selected_params.get('high_hu_threshold'),
        low_hu_threshold=tuning.selected_params.get('low_hu_threshold'),
    )

uncertainty = make_tpv_uncertainty_summary(reports, tuned_refined, core_results)

for row in uncertainty.rows():
    print(row)

final_report_path = write_html_report(
    SAVE_DIR / 'openplaque_tuned_tpv_report.html',
    reports,
    tuned_refined,
    core_results,
    uncertainty,
    title='OpenPlaque Tuned TPV Boundary Refinement Report',
)
print('Saved:', final_report_path)


In [ ]:
# 17. Optional visual QC of tuned masks
for report in reports:
    print('\n' + '='*80)
    print(report.name)
    tuned_refined[report.name].summary()
    tuned_refined[report.name].show_refined_overlay(report.volume)
    tuned_refined[report.name].show_removed_overlay(report.volume)


In [ ]:
# 18. Save tuned refined masks as NIfTI
import SimpleITK as sitk

def save_refined_mask(ref, reference_image, path):
    img = sitk.GetImageFromArray(ref.refined_mask.astype('uint8'))
    img.CopyInformation(reference_image)
    sitk.WriteImage(img, str(path))
    print('Saved:', path)

for report in reports:
    save_refined_mask(
        tuned_refined[report.name],
        report.mask_image,
        SAVE_DIR / f'{report.name}_tuned_refined_plaque_segmentation.nii.gz',
    )
